# Indicator 6.1.1 — Global Building Exposure to Wildfire-Affected Areas

This notebook estimates the **number and percentage of GHS-OBAT buildings whose representative longitude/latitude point falls within an area mapped as burned by GABAM 2024**.

The indicator measures **wildfire exposure**, not confirmed building destruction, damage, vulnerability or risk. A building is classified as exposed when its GHS-OBAT point samples a valid GABAM raster value greater than the configured burned-area threshold.

**Interpretation limits**

- GABAM maps annual **burned area**. Depending on location, mapped burning may include wildfires, prescribed fire or agricultural burning; the indicator therefore uses burned area as a wildfire-exposure proxy.
- GHS-OBAT represents a building by one point. A footprint that partly intersects burned area may be missed, while a point located on a burned pixel classifies the complete building record as exposed.
- GABAM 2024 is compared with the GHS-OBAT E2020 building stock, so the datasets are not temporally identical.
- Results describe exposure during the selected annual map only and do not represent cumulative historical exposure.

**Core datasets**

- **GABAM 2024 v3** — 30 m global annual burned-area map derived from Landsat imagery and distributed as 5° × 5° GeoTIFF tiles in `GABAM2024.zip`.
- **GHS-OBAT R2024A, epoch E2020** — country-level building records with representative longitude and latitude coordinates.
- **World Countries GeoJSON** — primary country/territory boundaries used to select the GABAM tiles required by each ISO3 code.
- **Natural Earth Admin-0 Map Units** — local fallback boundaries for ISO3 codes not resolved from the primary boundary file.

**Primary outputs for each ISO3 code**

- total building records;
- buildings with valid coordinates;
- buildings covered by required GABAM tile extents;
- buildings with a valid GABAM pixel value;
- buildings exposed to mapped burned area;
- exposure percentages using total buildings, valid-coordinate buildings and valid GABAM pixels as denominators;
- data-coverage and processing-quality fields.

GABAM source record: `https://zenodo.org/records/17707433`  
DOI: `10.5281/zenodo.17707433`

## Global map outputs in this version

Validated country CSVs can be combined into:

- a cumulative global CSV;
- a GeoPackage containing country geometries and indicator fields;
- an interactive HTML choropleth;
- a static PNG map.

### V3 reliability fix

The batch and dashboard runners now preserve country-level failure details and skip global-map generation safely when no validated country result exists.


## Notebook map

1. Import packages and check the environment  
2. Configure the workflow  
3. Load the ISO3 country and territory list  
4. Configure the official data repositories  
5. Resolve and download the GABAM 2024 archive  
6. Inspect the archive and build the GABAM tile catalogue  
7. Load the Esri World Countries boundary dataset  
8. Create one country-level boundary for each ISO3 code  
9. Complete missing boundaries using Natural Earth  
10. Identify the GABAM tiles required by a country  
11. Extract and validate required GABAM tiles  
12. Validate GABAM raster structure and coordinate systems  
13. Inspect GABAM pixel-value encoding  
14. Resolve the GHS-OBAT country file  
15. Download the GHS-OBAT country file when needed  
16. Inspect the GHS-OBAT compressed CSV  
17. Test GABAM sampling and wildfire-exposure logic  
18. Process GHS-OBAT buildings in chunks with checkpoints  
19. Run an optional complete single-country test  
20. Validate and protect the country result  
21. Record outcomes and quarantine invalid files  
22. Prepare country-processing and cleanup helpers  
23. Process one country from start to finish  
24. Run a selected multi-country sequence  
25. Prepare test or global execution  
26. Run all countries in automatic batches  
27. Generate global exposure maps  
28. Monitor live global-processing progress  
29. Interactive workflow dashboard  
30. Start or display the workflow controls


## General workflow

For each ISO3 code, the workflow:

1. resolves one country/territory boundary;
2. intersects that boundary with a catalogue of 5° × 5° GABAM tiles;
3. downloads the global GABAM ZIP once and extracts only the required GeoTIFF tiles to a shared cache;
4. resolves the official GHS-OBAT country ZIP;
5. streams all GHS-OBAT CSV parts in chunks without extracting them;
6. validates longitude and latitude values;
7. samples the required GABAM rasters at each valid building point;
8. classifies a building as exposed when the sampled value is valid and greater than `BURNED_VALUE_THRESHOLD`;
9. writes one validated country CSV and a resumable checkpoint;
10. appends the outcome to a processing log;
11. optionally removes country-specific raw data while retaining shared GABAM files for reuse.

### Exposure equations

For country or territory \(c\):

\[
E_c = \sum_{i=1}^{N_c} I(v_i \text{ is valid and } v_i > T)
\]

where \(v_i\) is the sampled GABAM value for building \(i\), \(T\) is `BURNED_VALUE_THRESHOLD`, and \(I\) is an indicator function.

The primary total-building percentage is:

\[
P_c = 100 \times \frac{E_c}{N_c}
\]

Additional percentages are calculated against buildings with valid coordinates and buildings with valid GABAM pixels so data coverage remains traceable.


## Processing log

Every country attempt writes a row to a year-specific CSV log. The log records:

- ISO3 code and analysis year;
- start and finish timestamps;
- processing duration;
- status and processing stage;
- output path;
- building, coordinate, tile-coverage, raster-valid and exposure counts;
- exposure and coverage percentages;
- warning or error text.

Expected non-error skips, such as a missing GHS-OBAT file or a country outside GABAM's published latitude coverage, are recorded separately from failed runs.


## 1. Import packages and check the environment

The first cells verify that the packages required by the complete workflow are importable before any large file is downloaded or processed.


In [1]:
from importlib import import_module
import platform
import sys

REQUIRED_PACKAGES = {
    "geopandas": "geopandas",
    "numpy": "numpy",
    "pandas": "pandas",
    "rasterio": "rasterio",
    "requests": "requests",
    "folium": "folium",
    "matplotlib": "matplotlib",
    "branca": "branca",
    "ipywidgets": "ipywidgets",
}

package_records = []

for package_name, module_name in REQUIRED_PACKAGES.items():
    try:
        module = import_module(module_name)
        version = getattr(module, "__version__", "unknown")
        package_records.append(
            {
                "package": package_name,
                "available": True,
                "version": version,
                "error": "",
            }
        )
    except Exception as error:
        package_records.append(
            {
                "package": package_name,
                "available": False,
                "version": "",
                "error": str(error),
            }
        )

missing_packages = [
    row["package"]
    for row in package_records
    if not row["available"]
]

print("Python:", sys.version)
print("Platform:", platform.platform())

if missing_packages:
    raise ImportError(
        "Missing required packages: "
        f"{missing_packages}. Install them before continuing."
    )

print("All required packages are available.")


Python: 3.12.11 | packaged by conda-forge | (main, Jun  4 2025, 14:45:31) [GCC 13.3.0]
Platform: Linux-6.8.0-1059-azure-x86_64-with-glibc2.39
All required packages are available.


In [2]:
import gc
import hashlib
import html
import json
import shutil
import traceback
from contextlib import ExitStack
from datetime import datetime, timezone
from math import ceil, floor
from pathlib import Path
from time import perf_counter, sleep
from urllib.parse import urljoin
from zipfile import BadZipFile, ZipFile

import folium
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import requests
from branca.colormap import linear
from IPython.display import HTML, display
from rasterio.crs import CRS
from rasterio.enums import Resampling
from rasterio.warp import transform as transform_coordinates
from shapely.geometry import box


## 2. Configure the workflow

This section defines the settings used by every later step.

### Key settings reference

| Setting | Purpose |
|---|---|
| `ANALYSIS_YEAR` | Selects the GABAM annual burned-area map. This notebook is configured for 2024. |
| `TEST_MODE` | Restricts processing to `TEST_COUNTRIES`. |
| `USE_WIDGET_DASHBOARD` | Displays controls at the end and prevents automatic execution. |
| `BURNED_VALUE_THRESHOLD` | A valid GABAM value greater than this threshold is treated as burned. |
| `ALLOW_NON_BINARY_GABAM_VALUES` | Allows processing when observed values are not limited to 0 and 1. Keep `False` until encoding is verified. |
| `CHUNK_SIZE` | Number of GHS-OBAT rows read per chunk. |
| `ENABLE_CHECKPOINTS` | Saves resumable country progress. |
| `MIN_TILE_COVERAGE_PERCENT` | Quality threshold for valid building coordinates falling within extracted GABAM tile extents. |
| `SKIP_COMPLETED` | Reuses a valid existing country result. |
| `KEEP_GABAM_ARCHIVE` | Retains the downloaded global GABAM ZIP. |
| `KEEP_GABAM_TILE_CACHE` | Retains extracted shared GABAM tiles for neighbouring countries. |
| `KEEP_BUILDING_RAW_DATA` | Retains country-specific GHS-OBAT ZIPs after successful processing. |
| `GENERATE_GLOBAL_MAPS` | Creates enabled global CSV/map products after processing. |


In [3]:
# Coordinate systems and dataset versions
OBAT_COORDINATE_CRS = CRS.from_epsg(4326)
GABAM_COORDINATE_CRS = CRS.from_epsg(4326)

ANALYSIS_YEAR = 2024
AVAILABLE_GABAM_YEARS = [2024]

if ANALYSIS_YEAR not in AVAILABLE_GABAM_YEARS:
    raise ValueError(
        f"ANALYSIS_YEAR must be one of {AVAILABLE_GABAM_YEARS}."
    )

GABAM_VERSION = "v3"
GABAM_RECORD_ID = 17707433
GABAM_ARCHIVE_FILENAME = f"GABAM{ANALYSIS_YEAR}.zip"
GABAM_ARCHIVE_MD5 = "b6c4b70bc95f9c3c30f4dfd5505ef12d"
GABAM_ARCHIVE_INNER_DIRECTORY = (
    f"ba_tiles-{ANALYSIS_YEAR}-30m-prob_S90G50M11"
)
GABAM_VRT_FILENAME = f"gabam{ANALYSIS_YEAR}.vrt"
GABAM_TILE_SIZE_DEGREES = 5
GABAM_COVERAGE_LAT_MIN = -60
GABAM_COVERAGE_LAT_MAX = 80
GABAM_COVERAGE_GEOMETRY = box(
    -180,
    GABAM_COVERAGE_LAT_MIN,
    180,
    GABAM_COVERAGE_LAT_MAX,
)

# Exposure interpretation
BURNED_VALUE_THRESHOLD = 0
EXPECTED_BINARY_GABAM_VALUES = {0, 1}
ALLOW_NON_BINARY_GABAM_VALUES = False
ADDITIONAL_GABAM_NODATA_VALUES: set[float] = set()

# Processing mode
TEST_MODE = True
TEST_COUNTRIES = ["LUX"]
USE_WIDGET_DASHBOARD = True

# Optional tests; these run only in TEST_MODE
RUN_PARTIAL_TEST = False
RUN_FULL_COUNTRY_TEST = False
RUN_PROCESS_COUNTRY_TEST = False
RUN_SMALL_MULTI_COUNTRY_TEST = False
PARTIAL_TEST_MAX_CHUNKS = 2

RUN_DIRECT_ANALYSIS_TESTS = (
    TEST_MODE
    and (RUN_PARTIAL_TEST or RUN_FULL_COUNTRY_TEST)
)

ANY_TEST_ENABLED = (
    TEST_MODE
    and (
        RUN_PARTIAL_TEST
        or RUN_FULL_COUNTRY_TEST
        or RUN_PROCESS_COUNTRY_TEST
        or RUN_SMALL_MULTI_COUNTRY_TEST
    )
)

# Chunking, checkpointing and quality controls
CHUNK_SIZE = 250_000
ENABLE_CHECKPOINTS = True
CHECKPOINT_EVERY_CHUNKS = 1
MIN_TILE_COVERAGE_PERCENT = 99.0
FAIL_ON_LOW_TILE_COVERAGE = False
SKIP_COMPLETED = True

if CHECKPOINT_EVERY_CHUNKS < 1:
    raise ValueError("CHECKPOINT_EVERY_CHUNKS must be at least 1.")

if not 0 <= MIN_TILE_COVERAGE_PERCENT <= 100:
    raise ValueError(
        "MIN_TILE_COVERAGE_PERCENT must be between 0 and 100."
    )

# Data-retention behaviour
KEEP_GABAM_ARCHIVE = True
KEEP_GABAM_TILE_CACHE = True
KEEP_BUILDING_RAW_DATA = False

# Global outputs
GENERATE_GLOBAL_MAPS = True
GENERATE_GLOBAL_INTERACTIVE_HTML = True
GENERATE_GLOBAL_STATIC_PNG = True
SAVE_GLOBAL_GPKG = True
GLOBAL_MAP_BASE_TILES = "CartoDB positron"
GLOBAL_MAP_PERCENT_BINS = [0, 0.1, 0.5, 1, 2, 5, 10, 25, 100]

if (
    GLOBAL_MAP_PERCENT_BINS[0] != 0
    or GLOBAL_MAP_PERCENT_BINS[-1] != 100
    or GLOBAL_MAP_PERCENT_BINS
    != sorted(set(GLOBAL_MAP_PERCENT_BINS))
):
    raise ValueError(
        "GLOBAL_MAP_PERCENT_BINS must be unique, sorted and span 0–100."
    )

# Network settings
REQUEST_TIMEOUT = 600
DOWNLOAD_CHUNK_SIZE = 1024 * 1024
MAX_DOWNLOAD_RETRIES = 3
REMOTE_CHECK_RETRY_DELAY_SECONDS = 5

# Project directories
# Use the same fixed project root as the flood notebook.
# Fall back to the notebook working directory only outside JupyterHub.
DEFAULT_PROJECT_DIR = Path("/home/jovyan/BIT")
PROJECT_DIR = (
    DEFAULT_PROJECT_DIR
    if DEFAULT_PROJECT_DIR.exists()
    else Path.cwd()
).resolve()

DATA_DIR = PROJECT_DIR / "data"
GABAM_DATA_DIR = DATA_DIR / "gabam" / str(ANALYSIS_YEAR)
GABAM_ARCHIVE_DIR = GABAM_DATA_DIR / "archive"
GABAM_TILE_CACHE_DIR = GABAM_DATA_DIR / "shared_tiles"
GABAM_CHECKSUM_MARKER_PATH = (
    GABAM_ARCHIVE_DIR / f".{GABAM_ARCHIVE_FILENAME}.md5_verified.json"
)
BUILDING_DATA_DIR = DATA_DIR / "ghs_obat"


def get_output_directory(year: int) -> Path:
    return PROJECT_DIR / "outputs" / f"wildfire_GABAM{year}"


def get_country_output_directory(year: int) -> Path:
    return get_output_directory(year) / "country_results"


def get_global_map_output_directory(year: int) -> Path:
    return get_output_directory(year) / "global_maps"


def get_invalid_output_directory(year: int) -> Path:
    return get_output_directory(year) / "invalid_country_results"


def get_checkpoint_directory(year: int) -> Path:
    return get_output_directory(year) / "checkpoints"


def get_log_directory(year: int) -> Path:
    return get_output_directory(year) / "logs"


def get_processing_log_path(year: int) -> Path:
    return get_log_directory(year) / f"processing_log_GABAM{year}.csv"


OUTPUT_DIR = get_output_directory(ANALYSIS_YEAR)
COUNTRY_OUTPUT_DIR = get_country_output_directory(ANALYSIS_YEAR)
GLOBAL_MAP_OUTPUT_DIR = get_global_map_output_directory(ANALYSIS_YEAR)
INVALID_OUTPUT_DIR = get_invalid_output_directory(ANALYSIS_YEAR)
CHECKPOINT_DIR = get_checkpoint_directory(ANALYSIS_YEAR)
LOG_DIR = get_log_directory(ANALYSIS_YEAR)
PROCESSING_LOG_PATH = get_processing_log_path(ANALYSIS_YEAR)

for directory in [
    DATA_DIR,
    GABAM_ARCHIVE_DIR,
    GABAM_TILE_CACHE_DIR,
    BUILDING_DATA_DIR,
    COUNTRY_OUTPUT_DIR,
    GLOBAL_MAP_OUTPUT_DIR,
    INVALID_OUTPUT_DIR,
    CHECKPOINT_DIR,
    LOG_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Analysis year:", ANALYSIS_YEAR)
print("GABAM version:", GABAM_VERSION)
print("Test mode:", TEST_MODE)
print("Selected test countries:", TEST_COUNTRIES)
print("Dashboard mode:", USE_WIDGET_DASHBOARD)
print("Any test enabled:", ANY_TEST_ENABLED)
print("Chunk size:", f"{CHUNK_SIZE:,}")
print("Burned-value threshold:", BURNED_VALUE_THRESHOLD)
print("Allow non-binary GABAM values:", ALLOW_NON_BINARY_GABAM_VALUES)
print("Minimum tile coverage:", MIN_TILE_COVERAGE_PERCENT)
print("Project directory:", PROJECT_DIR)
print("Output directory:", OUTPUT_DIR)
print("Processing log:", PROCESSING_LOG_PATH)

# S3 dataset storage settings
S3_BUCKET = "bit-alpha-data-802892343761-a955a2d204b1-indicator-6-7491703904"
ENABLE_S3_SYNC = True


Analysis year: 2024
GABAM version: v3
Test mode: True
Selected test countries: ['LUX']
Dashboard mode: True
Any test enabled: False
Chunk size: 250,000
Burned-value threshold: 0
Allow non-binary GABAM values: False
Minimum tile coverage: 99.0
Project directory: /home/jovyan/BIT
Output directory: /home/jovyan/BIT/outputs/wildfire_GABAM2024
Processing log: /home/jovyan/BIT/outputs/wildfire_GABAM2024/logs/processing_log_GABAM2024.csv


## 3. Load the ISO3 country and territory list

The workflow reads ISO 3166-1 alpha-3 codes from `countries.csv`, automatically detects a supported ISO3 field, removes malformed/duplicate values and selects either the configured test list or the complete cleaned list.


In [4]:
COUNTRY_LIST_PATH = PROJECT_DIR / "countries.csv"

if not COUNTRY_LIST_PATH.exists():
    raise FileNotFoundError(
        f"Country list not found: {COUNTRY_LIST_PATH}"
    )

countries_table = pd.read_csv(COUNTRY_LIST_PATH)

candidate_iso3_columns = [
    "alpha-3",
    "iso3",
    "ISO3",
    "iso_alpha3",
    "ISO_ALPHA_3",
]

ISO3_COLUMN = next(
    (
        column
        for column in candidate_iso3_columns
        if column in countries_table.columns
    ),
    None,
)

if ISO3_COLUMN is None:
    raise ValueError(
        "No supported ISO3 column was found. "
        f"Available columns: {countries_table.columns.tolist()}"
    )

ALL_COUNTRIES = (
    countries_table[ISO3_COLUMN]
    .dropna()
    .astype(str)
    .str.strip()
    .str.upper()
    .loc[lambda codes: codes.str.fullmatch(r"[A-Z]{3}")]
    .drop_duplicates()
    .tolist()
)

if TEST_MODE:
    invalid_test_codes = [
        iso3 for iso3 in TEST_COUNTRIES if iso3 not in ALL_COUNTRIES
    ]
    if invalid_test_codes:
        raise ValueError(
            "The following test codes are not present in countries.csv: "
            f"{invalid_test_codes}"
        )
    RUN_COUNTRIES = TEST_COUNTRIES.copy()
else:
    RUN_COUNTRIES = ALL_COUNTRIES.copy()

print("Detected ISO3 column:", ISO3_COLUMN)
print("Valid ISO3 codes loaded:", len(ALL_COUNTRIES))
print("Codes selected for this run:", len(RUN_COUNTRIES))
print("Selected codes:", RUN_COUNTRIES[:20])


Detected ISO3 column: alpha-3
Valid ISO3 codes loaded: 249
Codes selected for this run: 1
Selected codes: ['LUX']


## 4. Configure the official data repositories

The GABAM dataset is distributed through one global Zenodo ZIP for the selected year. GHS-OBAT continues to use the same official JRC country-file repository as the flood notebook.


In [5]:
GABAM_RECORD_URL = (
    f"https://zenodo.org/records/{GABAM_RECORD_ID}"
)
GABAM_ARCHIVE_URL = (
    f"{GABAM_RECORD_URL}/files/"
    f"{GABAM_ARCHIVE_FILENAME}?download=1"
)

GHS_OBAT_BASE_URL = (
    "https://jeodpp.jrc.ec.europa.eu/ftp/"
    "jrc-opendata/GHSL/GHS_OBAT_GLOBE_R2024A/"
    "GHS_OBAT_CSV_GLOBE_R2024A/V1-0/"
)

SOURCE_URLS = {
    "GABAM Zenodo record": GABAM_RECORD_URL,
    "GABAM global archive": GABAM_ARCHIVE_URL,
    "GHS-OBAT country files": GHS_OBAT_BASE_URL,
}

print("GABAM record:", GABAM_RECORD_URL)
print("GABAM archive:", GABAM_ARCHIVE_URL)
print("GHS-OBAT directory:", GHS_OBAT_BASE_URL)


GABAM record: https://zenodo.org/records/17707433
GABAM archive: https://zenodo.org/records/17707433/files/GABAM2024.zip?download=1
GHS-OBAT directory: https://jeodpp.jrc.ec.europa.eu/ftp/jrc-opendata/GHSL/GHS_OBAT_GLOBE_R2024A/GHS_OBAT_CSV_GLOBE_R2024A/V1-0/


### Informational repository preflight

A failed preflight produces a warning rather than stopping the notebook. Later file-level functions validate local caches first and retry network requests only when a required file is missing.


In [6]:
def check_remote_file(
    url: str,
    timeout: int = 60,
) -> dict:
    """Check remote accessibility without intentionally downloading a file."""
    last_error = None

    for attempt in range(1, MAX_DOWNLOAD_RETRIES + 1):
        try:
            # Range requests normally return only one byte while exposing
            # Content-Range with the complete file size.
            response = requests.get(
                url,
                headers={"Range": "bytes=0-0"},
                stream=True,
                timeout=timeout,
            )
            status_code = response.status_code
            content_range = response.headers.get("Content-Range", "")
            content_length = response.headers.get("Content-Length")
            response.close()

            accessible = status_code in {200, 206}
            file_size_bytes = None

            if "/" in content_range:
                total_text = content_range.rsplit("/", 1)[-1]
                if total_text.isdigit():
                    file_size_bytes = int(total_text)
            elif content_length and content_length.isdigit():
                file_size_bytes = int(content_length)

            return {
                "status_code": status_code,
                "accessible": accessible,
                "file_size_bytes": file_size_bytes,
                "error": "" if accessible else f"HTTP {status_code}",
            }

        except requests.RequestException as error:
            last_error = error
            if attempt < MAX_DOWNLOAD_RETRIES:
                sleep(REMOTE_CHECK_RETRY_DELAY_SECONDS)

    return {
        "status_code": None,
        "accessible": False,
        "file_size_bytes": None,
        "error": str(last_error),
    }


connection_records = []

for source_name, source_url in SOURCE_URLS.items():
    result = check_remote_file(source_url, timeout=30)
    connection_records.append(
        {
            "source": source_name,
            **result,
        }
    )

connection_table = pd.DataFrame(connection_records)
display(connection_table)

if not connection_table["accessible"].all():
    print(
        "Warning: one or more repository preflight checks failed. "
        "The workflow will still use valid local caches where available."
    )


,source,status_code,accessible,file_size_bytes,error
0,GABAM Zenodo record,200,True,NaN,
1,GABAM global archive,206,True,900429311.0,
2,GHS-OBAT country files,206,True,3656.0,


## 5. Resolve and download the GABAM 2024 archive

GABAM 2024 is supplied as one approximately 900 MB global ZIP. The workflow:

- reuses a valid cached archive;
- checks ZIP structure and required internal files;
- verifies the official MD5 checksum;
- checks free disk space before downloading;
- streams to a `.part` file and retries failed requests.


In [7]:
def download_file(
    url: str,
    output_path: Path,
    overwrite: bool = False,
) -> Path:
    """Stream a download with retry and .part-file protection."""
    output_path.parent.mkdir(parents=True, exist_ok=True)

    if (
        output_path.exists()
        and output_path.stat().st_size > 0
        and not overwrite
    ):
        print("Reusing existing file:", output_path)
        return output_path

    temporary_path = Path(str(output_path) + ".part")
    temporary_path.unlink(missing_ok=True)
    last_error = None

    for attempt in range(1, MAX_DOWNLOAD_RETRIES + 1):
        try:
            print(
                f"Download attempt {attempt}/{MAX_DOWNLOAD_RETRIES}: "
                f"{output_path.name}"
            )
            with requests.get(
                url,
                stream=True,
                timeout=REQUEST_TIMEOUT,
            ) as response:
                response.raise_for_status()
                with temporary_path.open("wb") as destination:
                    for chunk in response.iter_content(
                        chunk_size=DOWNLOAD_CHUNK_SIZE
                    ):
                        if chunk:
                            destination.write(chunk)

            if temporary_path.stat().st_size == 0:
                raise IOError("The downloaded file is empty.")

            temporary_path.replace(output_path)
            print("Saved:", output_path)
            return output_path

        except Exception as error:
            last_error = error
            temporary_path.unlink(missing_ok=True)
            print(f"Attempt {attempt} failed: {error}")
            if attempt < MAX_DOWNLOAD_RETRIES:
                sleep(REMOTE_CHECK_RETRY_DELAY_SECONDS)

    raise RuntimeError(
        f"Download failed after {MAX_DOWNLOAD_RETRIES} attempts: {url}"
    ) from last_error


def calculate_md5(file_path: Path, block_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.md5()
    with file_path.open("rb") as file:
        while True:
            block = file.read(block_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def get_gabam_archive_path(year: int = ANALYSIS_YEAR) -> Path:
    return GABAM_ARCHIVE_DIR / f"GABAM{year}.zip"


def validate_local_gabam_archive(
    archive_path: Path,
) -> tuple[bool, str]:
    """Perform a lightweight structural check without re-hashing 900 MB."""
    if not archive_path.exists():
        return False, "file does not exist"
    if archive_path.stat().st_size == 0:
        return False, "file is empty"

    try:
        with ZipFile(archive_path, "r") as archive:
            names = archive.namelist()
            tif_members = [
                name
                for name in names
                if name.lower().endswith(".tif")
                and GABAM_ARCHIVE_INNER_DIRECTORY in name
            ]
            vrt_members = [
                name
                for name in names
                if Path(name).name.lower() == GABAM_VRT_FILENAME.lower()
            ]

            if not tif_members:
                return False, "ZIP contains no GABAM GeoTIFF tiles"
            if not vrt_members:
                return False, "ZIP does not contain the expected GABAM VRT"

            with archive.open(tif_members[0]) as source:
                source.read(1)

    except (BadZipFile, OSError, RuntimeError) as error:
        return False, str(error)

    return True, ""


def verify_gabam_archive_checksum(
    archive_path: Path,
    force: bool = False,
) -> tuple[bool, str]:
    """Verify the official MD5 once and cache the file signature."""
    if not archive_path.exists():
        return False, "file does not exist"

    stat = archive_path.stat()
    expected_marker = {
        "expected_md5": GABAM_ARCHIVE_MD5.lower(),
        "size_bytes": stat.st_size,
        "mtime_ns": stat.st_mtime_ns,
    }

    if GABAM_CHECKSUM_MARKER_PATH.exists() and not force:
        try:
            marker = json.loads(
                GABAM_CHECKSUM_MARKER_PATH.read_text(encoding="utf-8")
            )
            if all(marker.get(key) == value for key, value in expected_marker.items()):
                return True, ""
        except Exception:
            GABAM_CHECKSUM_MARKER_PATH.unlink(missing_ok=True)

    print("Verifying GABAM archive MD5 (one-time full-file read)...")
    observed_md5 = calculate_md5(archive_path).lower()
    if observed_md5 != GABAM_ARCHIVE_MD5.lower():
        return (
            False,
            f"MD5 mismatch: expected {GABAM_ARCHIVE_MD5}, "
            f"observed {observed_md5}",
        )

    marker = {**expected_marker, "observed_md5": observed_md5}
    temporary_marker = Path(str(GABAM_CHECKSUM_MARKER_PATH) + ".part")
    temporary_marker.write_text(
        json.dumps(marker, indent=2),
        encoding="utf-8",
    )
    temporary_marker.replace(GABAM_CHECKSUM_MARKER_PATH)
    return True, ""


def resolve_gabam_archive() -> dict:
    archive_path = get_gabam_archive_path()
    local_valid, local_error = validate_local_gabam_archive(archive_path)
    checksum_valid = False
    checksum_error = ""
    if local_valid:
        checksum_valid, checksum_error = verify_gabam_archive_checksum(
            archive_path
        )

    if local_valid and checksum_valid:
        return {
            "filename": archive_path.name,
            "url": GABAM_ARCHIVE_URL,
            "accessible": True,
            "source": "local_cache",
            "local_path": str(archive_path),
            "file_size_mb": archive_path.stat().st_size / 1024**2,
            "error": "",
        }

    if archive_path.exists():
        reason = local_error or checksum_error
        print("Removing invalid cached GABAM archive:", reason)
        archive_path.unlink(missing_ok=True)
        GABAM_CHECKSUM_MARKER_PATH.unlink(missing_ok=True)

    remote = check_remote_file(GABAM_ARCHIVE_URL, timeout=60)
    size_bytes = remote["file_size_bytes"]

    return {
        "filename": archive_path.name,
        "url": GABAM_ARCHIVE_URL,
        "accessible": remote["accessible"],
        "source": "remote",
        "local_path": str(archive_path),
        "file_size_mb": (
            size_bytes / 1024**2 if size_bytes is not None else None
        ),
        "error": remote["error"],
    }


def download_gabam_archive(record: dict | None = None) -> Path:
    if record is None:
        record = resolve_gabam_archive()

    archive_path = Path(record["local_path"])
    local_valid, _ = validate_local_gabam_archive(archive_path)
    checksum_valid = False
    if local_valid:
        checksum_valid, _ = verify_gabam_archive_checksum(archive_path)
    if local_valid and checksum_valid:
        print("Using cached GABAM archive:", archive_path)
        return archive_path

    if not record["accessible"]:
        raise FileNotFoundError(
            f"GABAM archive is not accessible: {record['error']}"
        )

    expected_bytes = None
    if record.get("file_size_mb") is not None:
        expected_bytes = int(record["file_size_mb"] * 1024**2)

    if expected_bytes is not None:
        free_bytes = shutil.disk_usage(archive_path.parent).free
        required_bytes = int(expected_bytes * 1.15)
        if free_bytes < required_bytes:
            raise OSError(
                "Insufficient free disk space for the GABAM archive. "
                f"Required approximately {required_bytes / 1024**3:.2f} GB; "
                f"available {free_bytes / 1024**3:.2f} GB."
            )

    downloaded_path = download_file(
        GABAM_ARCHIVE_URL,
        archive_path,
    )

    valid, error = validate_local_gabam_archive(downloaded_path)
    if valid:
        valid, error = verify_gabam_archive_checksum(
            downloaded_path,
            force=True,
        )
    if not valid:
        downloaded_path.unlink(missing_ok=True)
        GABAM_CHECKSUM_MARKER_PATH.unlink(missing_ok=True)
        raise IOError(f"Downloaded GABAM archive failed validation: {error}")

    return downloaded_path


In [8]:
gabam_archive_record = resolve_gabam_archive()
display(pd.DataFrame([gabam_archive_record]))

GABAM_ARCHIVE_PATH = None

if RUN_DIRECT_ANALYSIS_TESTS:
    GABAM_ARCHIVE_PATH = download_gabam_archive(gabam_archive_record)
    print("Validated GABAM archive:", GABAM_ARCHIVE_PATH)


,filename,url,accessible,source,local_path,file_size_mb,error
0,GABAM2024.zip,https://zenodo.org/records/17707433/files/GABA...,True,local_cache,/home/jovyan/BIT/data/gabam/2024/archive/GABAM...,858.716308,


## 6. Inspect the archive and build the GABAM tile catalogue

The archive member names encode the southwest corner of each 5° × 5° tile, for example `N00E005.tif`. The workflow parses those names into a GeoDataFrame containing tile bounds and the corresponding archive member path.

No separate tile-index file is required.


In [9]:
import re

GABAM_TILE_PATTERN = re.compile(
    r"^(?P<lat_hemisphere>[NS])(?P<latitude>\d{2})"
    r"(?P<lon_hemisphere>[EW])(?P<longitude>\d{3})\.tif$",
    flags=re.IGNORECASE,
)


def parse_gabam_tile_name(tile_name: str) -> dict:
    match = GABAM_TILE_PATTERN.fullmatch(Path(tile_name).name)
    if match is None:
        raise ValueError(f"Unsupported GABAM tile name: {tile_name}")

    latitude = int(match.group("latitude"))
    longitude = int(match.group("longitude"))

    if match.group("lat_hemisphere").upper() == "S":
        latitude *= -1
    if match.group("lon_hemisphere").upper() == "W":
        longitude *= -1

    west = float(longitude)
    north = float(latitude)
    east = west + GABAM_TILE_SIZE_DEGREES
    south = north - GABAM_TILE_SIZE_DEGREES

    return {
        "tile_name": Path(tile_name).name,
        "west": west,
        "south": south,
        "east": east,
        "north": north,
        "geometry": box(west, south, east, north),
    }


def build_gabam_tile_catalogue(archive_path: Path) -> gpd.GeoDataFrame:
    valid, error = validate_local_gabam_archive(archive_path)
    if not valid:
        raise ValueError(f"Invalid GABAM archive: {error}")

    records = []
    with ZipFile(archive_path, "r") as archive:
        for member in archive.infolist():
            if member.is_dir() or not member.filename.lower().endswith(".tif"):
                continue
            if GABAM_ARCHIVE_INNER_DIRECTORY not in member.filename:
                continue

            parsed = parse_gabam_tile_name(member.filename)
            records.append(
                {
                    **parsed,
                    "archive_member": member.filename,
                    "compressed_size_mb": member.compress_size / 1024**2,
                    "uncompressed_size_mb": member.file_size / 1024**2,
                }
            )

    if not records:
        raise ValueError("No parsable GABAM GeoTIFF tiles were found.")

    catalogue = gpd.GeoDataFrame(
        records,
        geometry="geometry",
        crs="EPSG:4326",
    )

    duplicate_names = catalogue["tile_name"].duplicated(keep=False)
    if duplicate_names.any():
        raise ValueError(
            "Duplicate GABAM tile names were found: "
            f"{catalogue.loc[duplicate_names, 'tile_name'].tolist()}"
        )

    return catalogue.sort_values("tile_name").reset_index(drop=True)


In [10]:
gabam_tile_catalogue = None

if GABAM_ARCHIVE_PATH is not None:
    gabam_tile_catalogue = build_gabam_tile_catalogue(
        GABAM_ARCHIVE_PATH
    )
    print("GABAM tile count:", len(gabam_tile_catalogue))
    print("Catalogue CRS:", gabam_tile_catalogue.crs)
    display(gabam_tile_catalogue.head())


## 7. Load the Esri World Countries boundary dataset

The same local `World_Countries.geojson` used by the flood notebook is the primary country/territory boundary source. It must define a CRS and is converted to EPSG:4326.


In [11]:
ESRI_BOUNDARY_PATH = PROJECT_DIR / "World_Countries.geojson"

if not ESRI_BOUNDARY_PATH.exists():
    raise FileNotFoundError(
        f"Boundary file not found: {ESRI_BOUNDARY_PATH}"
    )

esri_boundaries = gpd.read_file(ESRI_BOUNDARY_PATH)

if esri_boundaries.crs is None:
    raise ValueError(
        "The boundary dataset does not contain CRS information."
    )

esri_boundaries = esri_boundaries.to_crs("EPSG:4326")

print("Boundary file:", ESRI_BOUNDARY_PATH)
print("Coordinate system:", esri_boundaries.crs)
print("Number of boundary records:", len(esri_boundaries))
print("Columns:", esri_boundaries.columns.tolist())
display(esri_boundaries.head())


Boundary file: /home/jovyan/BIT/World_Countries.geojson
Coordinate system: EPSG:4326
Number of boundary records: 651
Columns: ['FID', 'COUNTRY', 'ISO_CC', 'CONTINENT', 'LAND_TYPE', 'LAND_RANK', 'COUNTRYAFF', 'geometry']


,FID,COUNTRY,ISO_CC,CONTINENT,LAND_TYPE,LAND_RANK,COUNTRYAFF,geometry
0,1,Afghanistan,AFG,Asia,Primary land,5,Afghanistan,"POLYGON ((70.99982 38.47927, 71.00838 38.4706,..."
1,2,Albania,ALB,Europe,Primary land,5,Albania,"POLYGON ((19.74991 42.62826, 19.74991 42.6254,..."
2,3,Albania,ALB,Europe,Very small island,1,Albania,"MULTIPOLYGON (((19.99205 39.77532, 19.99213 39..."
3,4,Albania,ALB,Europe,Small island,2,Albania,"MULTIPOLYGON (((19.28039 40.50329, 19.28319 40..."
4,5,Algeria,DZA,Africa,Small island,2,Algeria,"POLYGON ((-1.12906 35.7246, -1.12938 35.72098,..."


## 9. Standardize and dissolve country boundaries

The boundary file uses the `ISO_CC` field. Although the field name is generic, its values in `World_Countries.geojson` are ISO3 codes such as `AFG`, `ALB` and `LUX`. This step validates the actual values, keeps valid ISO3 records, repairs geometries and dissolves separated polygons to one country-level geometry per ISO3 code.


In [12]:
BOUNDARY_ISO3_FIELD = "ISO_CC"

required_boundary_columns = {
    BOUNDARY_ISO3_FIELD,
    "COUNTRY",
    "CONTINENT",
    "geometry",
}

missing_boundary_columns = (
    required_boundary_columns - set(esri_boundaries.columns)
)

if missing_boundary_columns:
    raise ValueError(
        "The boundary dataset is missing required columns: "
        f"{sorted(missing_boundary_columns)}"
    )

# ISO_CC is named generically, but in this boundary file its values are
# three-letter ISO3 codes such as AFG, ALB and LUX. Validate the values
# rather than inferring the code type from the column name.
raw_boundary_codes = (
    esri_boundaries[BOUNDARY_ISO3_FIELD]
    .astype("string")
    .str.strip()
    .str.upper()
)

code_length_counts = (
    raw_boundary_codes.dropna().str.len().value_counts().sort_index()
)
valid_iso3_count = int(
    raw_boundary_codes.str.fullmatch(r"[A-Z]{3}", na=False).sum()
)

print("Boundary file:", ESRI_BOUNDARY_PATH.resolve())
print("Boundary code field:", BOUNDARY_ISO3_FIELD)
print("Boundary code length counts:", code_length_counts.to_dict())
print("Valid ISO3-formatted source records:", valid_iso3_count)
print("Sample source codes:", raw_boundary_codes.dropna().head(10).tolist())

if valid_iso3_count == 0:
    raise ValueError(
        "ISO_CC does not contain any valid three-letter ISO3 codes. "
        "Inspect the printed sample values before continuing."
    )

boundary_records = esri_boundaries[
    [
        BOUNDARY_ISO3_FIELD,
        "COUNTRY",
        "CONTINENT",
        "geometry",
    ]
].copy()

boundary_records["iso3"] = (
    boundary_records[BOUNDARY_ISO3_FIELD]
    .astype("string")
    .str.strip()
    .str.upper()
)

boundary_records = boundary_records.loc[
    boundary_records["iso3"].str.fullmatch(r"[A-Z]{3}", na=False)
].copy()

boundary_records = boundary_records.dropna(subset=["geometry"])
boundary_records = boundary_records.loc[
    ~boundary_records.geometry.is_empty
].copy()

boundary_records["geometry"] = boundary_records.geometry.make_valid()

country_boundaries = (
    boundary_records
    .dissolve(
        by="iso3",
        as_index=False,
        aggfunc={
            "COUNTRY": "first",
            "CONTINENT": "first",
        },
    )
    .rename(
        columns={
            "COUNTRY": "country_name",
            "CONTINENT": "continent",
        }
    )
    .to_crs("EPSG:4326")
)

country_boundaries = country_boundaries[
    ["iso3", "country_name", "continent", "geometry"]
]

available_boundary_codes = set(country_boundaries["iso3"])
missing_boundary_codes = sorted(
    set(ALL_COUNTRIES) - available_boundary_codes
)
missing_run_boundary_codes = sorted(
    set(RUN_COUNTRIES) - available_boundary_codes
)

print("Original boundary records:", len(esri_boundaries))
print("Valid boundary records:", len(boundary_records))
print("Country-level geometries:", len(country_boundaries))
print("Records merged:", len(boundary_records) - len(country_boundaries))
print("ISO3 codes without a matched boundary:", len(missing_boundary_codes))
print(missing_boundary_codes)
print("Selected run codes without a boundary:", missing_run_boundary_codes)

display(country_boundaries.head())
print("Boundary preparation completed successfully.")


Boundary file: /home/jovyan/BIT/World_Countries.geojson
Boundary code field: ISO_CC
Boundary code length counts: {np.int64(3): 651}
Valid ISO3-formatted source records: 651
Sample source codes: ['AFG', 'ALB', 'ALB', 'ALB', 'DZA', 'DZA', 'DZA', 'ASM', 'ASM', 'ASM']
Original boundary records: 651
Valid boundary records: 651
Country-level geometries: 243
Records merged: 408
ISO3 codes without a matched boundary: 6
['ALA', 'ESH', 'HKG', 'MAC', 'SJM', 'TWN']
Selected run codes without a boundary: []


,iso3,country_name,continent,geometry
0,ABW,Aruba,South America,"POLYGON ((-69.98723 12.55956, -69.98366 12.557..."
1,AFG,Afghanistan,Asia,"POLYGON ((70.99982 38.47927, 71.00838 38.4706,..."
2,AGO,Angola,Africa,"MULTIPOLYGON (((11.77149 -17.25379, 11.76451 -..."
3,AIA,Anguilla,North America,"MULTIPOLYGON (((-63.18828 18.26627, -63.19194 ..."
4,ALB,Albania,Europe,"MULTIPOLYGON (((19.98992 39.77379, 19.9908 39...."


Boundary preparation completed successfully.


## 9. Complete missing boundaries using Natural Earth

The same local Natural Earth Admin-0 Map Units ZIP used by the flood workflow is applied only to unresolved ISO3 codes. If no codes are missing, the fallback file is not read.


In [13]:
NATURAL_EARTH_FALLBACK_PATH = (
    DATA_DIR / "boundaries" / "ne_10m_admin_0_map_units.zip"
)

if missing_boundary_codes:
    if not NATURAL_EARTH_FALLBACK_PATH.exists():
        raise FileNotFoundError(
            "Natural Earth fallback file is required for unresolved codes: "
            f"{NATURAL_EARTH_FALLBACK_PATH}"
        )

    natural_earth_boundaries = gpd.read_file(
        f"zip://{NATURAL_EARTH_FALLBACK_PATH.resolve()}"
    ).to_crs("EPSG:4326")

    fallback_iso3_fields = [
        field
        for field in ["ISO_A3", "ADM0_A3", "GU_A3", "SU_A3", "BRK_A3"]
        if field in natural_earth_boundaries.columns
    ]

    missing_code_set = set(missing_boundary_codes)

    def select_fallback_iso3(row):
        for field in fallback_iso3_fields:
            value = row.get(field)
            if pd.isna(value):
                continue
            value = str(value).strip().upper()
            if value in missing_code_set:
                return value
        return pd.NA

    natural_earth_boundaries["iso3"] = natural_earth_boundaries.apply(
        select_fallback_iso3,
        axis=1,
    )

    fallback_records = natural_earth_boundaries.dropna(
        subset=["iso3", "geometry"]
    ).copy()
    fallback_records["geometry"] = fallback_records.geometry.make_valid()

    fallback_name_field = next(
        (
            field
            for field in ["NAME", "ADMIN", "NAME_LONG"]
            if field in fallback_records.columns
        ),
        None,
    )
    if fallback_name_field is None:
        fallback_records["fallback_name"] = fallback_records["iso3"]
        fallback_name_field = "fallback_name"

    fallback_continent_field = (
        "CONTINENT" if "CONTINENT" in fallback_records.columns else None
    )
    fallback_agg = {fallback_name_field: "first"}
    if fallback_continent_field:
        fallback_agg[fallback_continent_field] = "first"

    fallback_country_boundaries = fallback_records.dissolve(
        by="iso3",
        as_index=False,
        aggfunc=fallback_agg,
    ).rename(columns={fallback_name_field: "country_name"})

    if fallback_continent_field:
        fallback_country_boundaries = fallback_country_boundaries.rename(
            columns={fallback_continent_field: "continent"}
        )
    else:
        fallback_country_boundaries["continent"] = pd.NA

    fallback_country_boundaries = fallback_country_boundaries[
        ["iso3", "country_name", "continent", "geometry"]
    ]

    country_boundaries = gpd.GeoDataFrame(
        pd.concat(
            [country_boundaries, fallback_country_boundaries],
            ignore_index=True,
        ),
        geometry="geometry",
        crs="EPSG:4326",
    )

country_boundaries = (
    country_boundaries
    .drop_duplicates(subset=["iso3"])
    .sort_values("iso3")
    .reset_index(drop=True)
)

remaining_missing_boundary_codes = sorted(
    set(ALL_COUNTRIES) - set(country_boundaries["iso3"])
)

print("Total country-level boundaries:", len(country_boundaries))
print("Remaining codes without boundaries:", remaining_missing_boundary_codes)


Total country-level boundaries: 249
Remaining codes without boundaries: []


## 10. Identify the GABAM tiles required by a country

A country's repaired boundary is intersected with the archive-derived 5° tile catalogue. Tiles that only touch the boundary along a zero-area edge are excluded. Countries wholly outside the published GABAM latitude range (80°N to 60°S) are treated as expected skips.


In [14]:
def get_country_boundary(iso3: str) -> gpd.GeoDataFrame:
    iso3 = iso3.strip().upper()
    selected = country_boundaries.loc[
        country_boundaries["iso3"] == iso3
    ].copy()
    if selected.empty:
        raise KeyError(f"No boundary was found for {iso3}.")
    return selected


def find_intersecting_gabam_tiles(
    iso3: str,
    tile_catalogue: gpd.GeoDataFrame,
) -> gpd.GeoDataFrame:
    boundary_row = get_country_boundary(iso3).iloc[0]
    boundary_geometry = boundary_row.geometry

    covered_geometry = boundary_geometry.intersection(
        GABAM_COVERAGE_GEOMETRY
    )
    if covered_geometry.is_empty:
        return tile_catalogue.iloc[0:0].copy()

    candidates = tile_catalogue.loc[
        tile_catalogue.intersects(covered_geometry)
    ].copy()

    if candidates.empty:
        return candidates

    positive_area = candidates.geometry.apply(
        lambda geometry: geometry.intersection(covered_geometry).area > 0
    )

    candidates = candidates.loc[positive_area].copy()
    candidates["iso3"] = iso3.strip().upper()
    candidates["country_name"] = boundary_row["country_name"]
    return candidates.sort_values("tile_name").reset_index(drop=True)


In [15]:
test_gabam_tiles = None

if RUN_DIRECT_ANALYSIS_TESTS:
    if gabam_tile_catalogue is None:
        gabam_tile_catalogue = build_gabam_tile_catalogue(
            GABAM_ARCHIVE_PATH
        )

    test_gabam_tiles = find_intersecting_gabam_tiles(
        TEST_COUNTRIES[0],
        gabam_tile_catalogue,
    )
    print("Required GABAM tiles:", len(test_gabam_tiles))
    display(
        test_gabam_tiles[
            ["tile_name", "west", "south", "east", "north", "archive_member"]
        ]
    )


## 11. Extract and validate required GABAM tiles

Required GeoTIFF members are copied from the global ZIP to a shared year-specific tile cache. Extraction uses `.part` protection, never trusts archive paths as output paths, and reopens each tile with Rasterio before accepting it.


In [16]:
def get_gabam_tile_path(tile_name: str) -> Path:
    return GABAM_TILE_CACHE_DIR / Path(tile_name).name


def validate_local_gabam_tile(tile_path: Path) -> tuple[bool, str]:
    if not tile_path.exists():
        return False, "file does not exist"
    if tile_path.stat().st_size == 0:
        return False, "file is empty"

    try:
        with rasterio.open(tile_path) as source:
            if source.count != 1:
                return False, f"expected 1 band, found {source.count}"
            if source.crs is None:
                return False, "CRS is undefined"
            if source.width < 1 or source.height < 1:
                return False, "raster dimensions are invalid"
            source.read(1, window=((0, 1), (0, 1)))
    except Exception as error:
        return False, str(error)

    return True, ""


def extract_gabam_tiles(
    archive_path: Path,
    tile_table: pd.DataFrame,
) -> list[Path]:
    if tile_table.empty:
        return []

    required_columns = {"tile_name", "archive_member"}
    missing = required_columns - set(tile_table.columns)
    if missing:
        raise ValueError(
            f"Tile table is missing required columns: {sorted(missing)}"
        )

    extracted_paths = []

    with ZipFile(archive_path, "r") as archive:
        archive_names = set(archive.namelist())

        for position, row in enumerate(
            tile_table.itertuples(index=False),
            start=1,
        ):
            output_path = get_gabam_tile_path(row.tile_name)
            local_valid, local_error = validate_local_gabam_tile(output_path)

            if local_valid:
                print(
                    f"[{position}/{len(tile_table)}] Using cached {row.tile_name}"
                )
                extracted_paths.append(output_path)
                continue

            if output_path.exists():
                print(
                    f"Removing invalid cached tile {row.tile_name}: {local_error}"
                )
                output_path.unlink(missing_ok=True)

            if row.archive_member not in archive_names:
                raise FileNotFoundError(
                    f"Archive member not found: {row.archive_member}"
                )

            temporary_path = Path(str(output_path) + ".part")
            temporary_path.unlink(missing_ok=True)
            output_path.parent.mkdir(parents=True, exist_ok=True)

            print(
                f"[{position}/{len(tile_table)}] Extracting {row.tile_name}"
            )
            with archive.open(row.archive_member) as source, temporary_path.open(
                "wb"
            ) as destination:
                shutil.copyfileobj(source, destination, length=8 * 1024 * 1024)

            temporary_path.replace(output_path)
            local_valid, local_error = validate_local_gabam_tile(output_path)
            if not local_valid:
                output_path.unlink(missing_ok=True)
                raise IOError(
                    f"Extracted GABAM tile failed validation: "
                    f"{row.tile_name}: {local_error}"
                )

            extracted_paths.append(output_path)

    return extracted_paths


In [17]:
test_gabam_tile_paths = []

if RUN_DIRECT_ANALYSIS_TESTS:
    test_gabam_tile_paths = extract_gabam_tiles(
        GABAM_ARCHIVE_PATH,
        test_gabam_tiles,
    )
    print("Validated extracted tiles:", len(test_gabam_tile_paths))


## 12. Validate GABAM raster structure and coordinate systems

All selected tiles must use one band, a defined common CRS and a consistent pixel size. GHS-OBAT coordinates are EPSG:4326; coordinate transformation is applied automatically if a future GABAM release uses another CRS.


In [18]:
def inspect_gabam_tiles(tile_paths: list[Path]) -> pd.DataFrame:
    if not tile_paths:
        raise ValueError("No GABAM tile paths were provided.")

    records = []
    for tile_path in tile_paths:
        valid, error = validate_local_gabam_tile(tile_path)
        if not valid:
            raise ValueError(f"Invalid GABAM tile {tile_path.name}: {error}")

        with rasterio.open(tile_path) as source:
            records.append(
                {
                    "filename": tile_path.name,
                    "crs": source.crs.to_string(),
                    "crs_epsg": source.crs.to_epsg(),
                    "data_type": source.dtypes[0],
                    "width": source.width,
                    "height": source.height,
                    "resolution_x": abs(source.res[0]),
                    "resolution_y": abs(source.res[1]),
                    "nodata": source.nodata,
                    "left": source.bounds.left,
                    "bottom": source.bounds.bottom,
                    "right": source.bounds.right,
                    "top": source.bounds.top,
                }
            )

    inspection = pd.DataFrame(records)

    if inspection["crs"].nunique() != 1:
        raise ValueError("Selected GABAM tiles do not share one CRS.")

    if len(inspection[["resolution_x", "resolution_y"]].drop_duplicates()) != 1:
        raise ValueError("Selected GABAM tiles do not share one resolution.")

    raster_crs = CRS.from_user_input(inspection["crs"].iloc[0])
    inspection["obat_coordinates_require_transform"] = (
        raster_crs != OBAT_COORDINATE_CRS
    )

    return inspection


In [19]:
test_gabam_tile_inspection = None

if RUN_DIRECT_ANALYSIS_TESTS:
    test_gabam_tile_inspection = inspect_gabam_tiles(
        test_gabam_tile_paths
    )
    display(test_gabam_tile_inspection)


## 13. Inspect GABAM pixel-value encoding

The archive folder name contains `prob`, so the notebook does not silently assume the raster is binary. A sample of valid values is inspected before country processing.

By default, only values `{0, 1}` are accepted. If the observed release uses another documented encoding, review the result and deliberately set `ALLOW_NON_BINARY_GABAM_VALUES=True` and an appropriate `BURNED_VALUE_THRESHOLD` before continuing.


In [20]:
def inspect_gabam_value_encoding(
    tile_paths: list[Path],
    maximum_tiles: int = 10,
    sample_dimension: int = 512,
) -> dict:
    """Inspect a decimated sample without loading full 5-degree tiles."""
    if not tile_paths:
        raise ValueError("No GABAM tiles were provided for value inspection.")

    observed_values = set()
    sampled_pixels = 0

    for tile_path in tile_paths[:maximum_tiles]:
        with rasterio.open(tile_path) as source:
            output_height = min(sample_dimension, source.height)
            output_width = min(sample_dimension, source.width)
            array = source.read(
                1,
                out_shape=(output_height, output_width),
                masked=True,
                resampling=Resampling.nearest,
            )
            compressed = np.asarray(array.compressed())

            # Note: GABAM declares nodata=0 in metadata, but 0 represents unburned land observations.
            # Both 0 (unburned) and 1 (burned) are valid data values.

            if ADDITIONAL_GABAM_NODATA_VALUES and compressed.size:
                additional_nodata = np.asarray(
                    sorted(ADDITIONAL_GABAM_NODATA_VALUES),
                    dtype=compressed.dtype,
                )
                compressed = compressed[
                    ~np.isin(compressed, additional_nodata)
                ]

            sampled_pixels += int(compressed.size)
            if compressed.size:
                observed_values.update(np.unique(compressed).tolist())

            if len(observed_values) > 100:
                break

    observed_values_sorted = sorted(observed_values)
    observed_value_set = set(observed_values_sorted)
    unexpected = observed_value_set - EXPECTED_BINARY_GABAM_VALUES

    result = {
        "sampled_tiles": min(len(tile_paths), maximum_tiles),
        "sampled_valid_pixels": sampled_pixels,
        "unique_value_count": len(observed_values_sorted),
        "minimum_value": (
            observed_values_sorted[0] if observed_values_sorted else np.nan
        ),
        "maximum_value": (
            observed_values_sorted[-1] if observed_values_sorted else np.nan
        ),
        "observed_values_preview": observed_values_sorted[:30],
        "unexpected_values_preview": sorted(unexpected)[:30],
        "binary_encoding_confirmed": not unexpected,
    }

    if unexpected and not ALLOW_NON_BINARY_GABAM_VALUES:
        raise ValueError(
            "Observed GABAM values are not limited to {0, 1}. "
            f"Unexpected values include {sorted(unexpected)[:30]}. "
            "Review the dataset encoding, NoData values and threshold "
            "before continuing."
        )

    return result


In [21]:
test_gabam_encoding = None

if RUN_DIRECT_ANALYSIS_TESTS:
    test_gabam_encoding = inspect_gabam_value_encoding(
        test_gabam_tile_paths
    )
    display(pd.DataFrame([test_gabam_encoding]))


## 14. Resolve the GHS-OBAT country file

GHS-OBAT R2024A is distributed as one country ZIP per available ISO3 code. The same filename, URL, local-cache and structural-validation logic used by the flood notebook is retained.


In [22]:
def build_ghs_obat_filename(iso3: str) -> str:
    iso3 = iso3.strip().upper()
    if not (len(iso3) == 3 and iso3.isalpha()):
        raise ValueError(f"Invalid ISO3 code: {iso3}")
    return f"GHS_OBAT_CSV_{iso3}_E2020_R2024A_V1_0.zip"


def build_ghs_obat_url(iso3: str) -> str:
    return urljoin(GHS_OBAT_BASE_URL, build_ghs_obat_filename(iso3))


def get_ghs_obat_zip_path(iso3: str) -> Path:
    iso3 = iso3.strip().upper()
    return BUILDING_DATA_DIR / iso3.lower() / build_ghs_obat_filename(iso3)


def validate_local_ghs_obat_zip(zip_path: Path) -> tuple[bool, str]:
    if not zip_path.exists():
        return False, "file does not exist"
    if zip_path.stat().st_size == 0:
        return False, "file is empty"

    try:
        with ZipFile(zip_path, "r") as archive:
            csv_members = [
                member
                for member in archive.infolist()
                if not member.is_dir()
                and member.filename.lower().endswith(".csv")
            ]
            if not csv_members:
                return False, "ZIP contains no CSV files"
            with archive.open(csv_members[0]) as csv_file:
                csv_file.read(1)
    except (BadZipFile, OSError, RuntimeError) as error:
        return False, str(error)

    return True, ""


def create_ghs_obat_download_record(iso3: str) -> dict:
    iso3 = iso3.strip().upper()
    filename = build_ghs_obat_filename(iso3)
    url = build_ghs_obat_url(iso3)
    local_path = get_ghs_obat_zip_path(iso3)

    local_valid, local_error = validate_local_ghs_obat_zip(local_path)
    if local_valid:
        return {
            "iso3": iso3,
            "filename": filename,
            "url": url,
            "status_code": None,
            "accessible": True,
            "file_size_bytes": local_path.stat().st_size,
            "file_size_mb": local_path.stat().st_size / 1024**2,
            "error": "",
            "source": "local_cache",
            "local_path": str(local_path),
        }

    if local_path.exists():
        print(f"Removing invalid cached GHS-OBAT file: {local_error}")
        local_path.unlink(missing_ok=True)

    check = check_remote_file(url, timeout=60)
    file_size_bytes = check["file_size_bytes"]

    return {
        "iso3": iso3,
        "filename": filename,
        "url": url,
        "status_code": check["status_code"],
        "accessible": check["accessible"],
        "file_size_bytes": file_size_bytes,
        "file_size_mb": (
            file_size_bytes / 1024**2 if file_size_bytes is not None else None
        ),
        "error": check["error"],
        "source": "remote",
        "local_path": str(local_path),
    }


In [23]:
test_obat_record = None

if RUN_DIRECT_ANALYSIS_TESTS:
    test_obat_record = create_ghs_obat_download_record(
        TEST_COUNTRIES[0]
    )
    display(pd.DataFrame([test_obat_record]))


## 15. Download the GHS-OBAT country file when needed

A valid cached ZIP is reused. Otherwise, the workflow checks available disk space and downloads the country file with the same `.part` and retry protection used for GABAM.


In [24]:
def download_ghs_obat_zip(
    iso3: str,
    download_record: dict | None = None,
) -> Path:
    if download_record is None:
        download_record = create_ghs_obat_download_record(iso3)

    output_path = Path(download_record["local_path"])

    # The country-specific cache directory may not exist yet, and it is
    # removed after successful processing when KEEP_BUILDING_RAW_DATA=False.
    # Create it before disk-space checks or downloading.
    output_path.parent.mkdir(parents=True, exist_ok=True)

    local_valid, _ = validate_local_ghs_obat_zip(output_path)
    if local_valid:
        print("Using cached GHS-OBAT file:", output_path)
        return output_path

    if not download_record["accessible"]:
        raise FileNotFoundError(
            f"GHS-OBAT is unavailable for {iso3}: "
            f"{download_record['error']}"
        )

    expected_bytes = download_record.get("file_size_bytes")
    if expected_bytes is not None:
        free_bytes = shutil.disk_usage(output_path.parent).free
        if free_bytes < int(expected_bytes * 1.2):
            raise OSError(
                f"Insufficient disk space for {output_path.name}."
            )

    downloaded = download_file(
        download_record["url"],
        output_path,
    )

    valid, error = validate_local_ghs_obat_zip(downloaded)
    if not valid:
        downloaded.unlink(missing_ok=True)
        raise IOError(
            f"Downloaded GHS-OBAT ZIP failed validation: {error}"
        )

    return downloaded


In [25]:
test_obat_zip_path = None

if RUN_DIRECT_ANALYSIS_TESTS:
    test_obat_zip_path = download_ghs_obat_zip(
        TEST_COUNTRIES[0],
        test_obat_record,
    )
    print("GHS-OBAT ZIP:", test_obat_zip_path)


## 16. Inspect the GHS-OBAT compressed CSV

The ZIP is opened without extracting its CSV members. The workflow reports all CSV parts, compressed/uncompressed size, columns and a small sample.

All building records are included regardless of `epoch`, height or use classification. The wildfire indicator uses only `lon` and `lat` for the spatial overlay.


In [26]:
def inspect_ghs_obat_zip(
    zip_path: Path,
    sample_rows: int = 5,
) -> tuple[dict, pd.DataFrame]:
    if not zip_path.exists():
        raise FileNotFoundError(f"GHS-OBAT ZIP not found: {zip_path}")

    with ZipFile(zip_path, "r") as archive:
        csv_members = sorted(
            [
                member
                for member in archive.infolist()
                if not member.is_dir()
                and member.filename.lower().endswith(".csv")
            ],
            key=lambda member: member.filename,
        )

        if not csv_members:
            raise ValueError(f"No CSV files were found in {zip_path.name}.")

        with archive.open(csv_members[0]) as csv_file:
            sample = pd.read_csv(
                csv_file,
                nrows=sample_rows,
                low_memory=False,
            )

        required_columns = {"lon", "lat"}
        missing = required_columns - set(sample.columns)
        if missing:
            raise ValueError(
                f"GHS-OBAT CSV is missing required columns: {sorted(missing)}"
            )

        metadata = {
            "zip_filename": zip_path.name,
            "csv_file_count": len(csv_members),
            "csv_filenames": [m.filename for m in csv_members],
            "compressed_size_mb": (
                sum(m.compress_size for m in csv_members) / 1024**2
            ),
            "uncompressed_size_gb": (
                sum(m.file_size for m in csv_members) / 1024**3
            ),
            "column_count": len(sample.columns),
            "columns": sample.columns.tolist(),
        }

    return metadata, sample


In [27]:
test_obat_metadata = None
test_obat_sample = None

if RUN_DIRECT_ANALYSIS_TESTS:
    test_obat_metadata, test_obat_sample = inspect_ghs_obat_zip(
        test_obat_zip_path
    )
    display(pd.DataFrame([test_obat_metadata]))
    display(test_obat_sample)


## 17. Test GABAM sampling and wildfire-exposure logic

The sampling function returns three arrays:

- sampled GABAM values;
- whether each coordinate lies within at least one selected tile extent;
- whether the sampled raster value is valid rather than NoData/masked.

A building is exposed only when the sampled value is valid and greater than `BURNED_VALUE_THRESHOLD`.


In [28]:
def sample_gabam_values_from_sources(
    tile_sources: list,
    longitude: np.ndarray,
    latitude: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    if not tile_sources:
        raise ValueError("At least one open GABAM raster is required.")

    longitude = np.asarray(longitude, dtype=np.float64)
    latitude = np.asarray(latitude, dtype=np.float64)

    if len(longitude) != len(latitude):
        raise ValueError(
            "Longitude and latitude arrays must have the same length."
        )

    source_crs_values = []
    for source in tile_sources:
        if source.crs is None:
            raise ValueError(
                f"GABAM raster has no CRS: {getattr(source, 'name', '<unknown>')}"
            )
        source_crs_values.append(source.crs.to_string())

    unique_crs = sorted(set(source_crs_values))
    if len(unique_crs) != 1:
        raise ValueError(
            f"All GABAM sources must share one CRS. Found: {unique_crs}"
        )

    raster_crs = CRS.from_user_input(unique_crs[0])
    point_count = len(longitude)
    values = np.full(point_count, np.nan, dtype=np.float32)
    tile_covered = np.zeros(point_count, dtype=bool)
    valid_pixel = np.zeros(point_count, dtype=bool)

    valid_input = (
        np.isfinite(longitude)
        & np.isfinite(latitude)
        & (longitude >= -180)
        & (longitude <= 180)
        & (latitude >= -90)
        & (latitude <= 90)
    )

    sample_x = np.full(point_count, np.nan, dtype=np.float64)
    sample_y = np.full(point_count, np.nan, dtype=np.float64)
    valid_indices = np.flatnonzero(valid_input)

    if valid_indices.size == 0:
        return values, tile_covered, valid_pixel

    if raster_crs == OBAT_COORDINATE_CRS:
        sample_x[valid_indices] = longitude[valid_indices]
        sample_y[valid_indices] = latitude[valid_indices]
    else:
        transformed_x, transformed_y = transform_coordinates(
            OBAT_COORDINATE_CRS,
            raster_crs,
            longitude[valid_indices].tolist(),
            latitude[valid_indices].tolist(),
        )
        sample_x[valid_indices] = np.asarray(transformed_x, dtype=np.float64)
        sample_y[valid_indices] = np.asarray(transformed_y, dtype=np.float64)

    transformed_valid = np.isfinite(sample_x) & np.isfinite(sample_y)

    for source in tile_sources:
        bounds = source.bounds
        inside = (
            transformed_valid
            & (sample_x >= bounds.left)
            & (sample_x < bounds.right)
            & (sample_y >= bounds.bottom)
            & (sample_y < bounds.top)
        )
        tile_covered |= inside

        candidate_indices = np.flatnonzero(
            inside & ~valid_pixel
        )
        if candidate_indices.size == 0:
            continue

        coordinates = zip(
            sample_x[candidate_indices],
            sample_y[candidate_indices],
        )
        # Note: GABAM rasters have nodata: 0 in metadata, but 0 is the unburned land observation.
        # Sample with masked=False to retain 0 (unburned) and 1 (burned) as valid data.
        sampled = list(source.sample(coordinates, indexes=1, masked=False))
        numeric_values = np.asarray(sampled, dtype=np.float32).reshape(-1)

        sample_valid = np.isfinite(numeric_values) & np.isin(numeric_values, [0.0, 1.0])

        if ADDITIONAL_GABAM_NODATA_VALUES:
            sample_valid &= ~np.isin(
                numeric_values,
                list(ADDITIONAL_GABAM_NODATA_VALUES),
            )

        values[candidate_indices[sample_valid]] = numeric_values[sample_valid]
        valid_pixel[candidate_indices[sample_valid]] = True

    return values, tile_covered, valid_pixel


def classify_wildfire_exposure(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=np.float32)
    return np.isfinite(values) & (values > BURNED_VALUE_THRESHOLD)


In [29]:
sample_test_result = None

if RUN_DIRECT_ANALYSIS_TESTS:
    with ExitStack() as stack:
        sources = [
            stack.enter_context(rasterio.open(path))
            for path in test_gabam_tile_paths
        ]

        valid_sample = test_obat_sample.dropna(subset=["lon", "lat"])
        sampled_values, tile_covered, valid_pixel = (
            sample_gabam_values_from_sources(
                sources,
                valid_sample["lon"].to_numpy(),
                valid_sample["lat"].to_numpy(),
            )
        )
        exposed = classify_wildfire_exposure(sampled_values)

        sample_test_result = valid_sample[["lon", "lat"]].copy()
        sample_test_result["gabam_value"] = sampled_values
        sample_test_result["tile_covered"] = tile_covered
        sample_test_result["valid_gabam_pixel"] = valid_pixel
        sample_test_result["wildfire_exposed"] = exposed

    display(sample_test_result)


## 18. Process GHS-OBAT buildings in chunks with checkpoints

National GHS-OBAT files may contain millions of records. CSV members are streamed directly from the ZIP in configured chunks. Checkpoints store completed CSV-part/chunk positions and cumulative counters so an interrupted country can resume without repeating completed chunks.


In [30]:
def find_obat_csv_members(zip_path: Path) -> list[str]:
    with ZipFile(zip_path, "r") as archive:
        members = sorted(
            [
                member.filename
                for member in archive.infolist()
                if not member.is_dir()
                and member.filename.lower().endswith(".csv")
            ]
        )
    if not members:
        raise ValueError(f"No CSV files found in {zip_path.name}.")
    return members


def calculate_percentage(numerator: int, denominator: int) -> float:
    if denominator <= 0:
        return np.nan
    return 100.0 * numerator / denominator


def get_country_checkpoint_path(iso3: str, year: int) -> Path:
    return get_checkpoint_directory(year) / f"{iso3}_GABAM{year}.json"


def build_country_checkpoint_signature(
    iso3: str,
    year: int,
    obat_zip_path: Path,
    tile_paths: list[Path],
    chunk_size: int,
) -> dict:
    return {
        "iso3": iso3,
        "analysis_year": year,
        "gabam_version": GABAM_VERSION,
        "gabam_archive_md5": GABAM_ARCHIVE_MD5,
        "burned_value_threshold": BURNED_VALUE_THRESHOLD,
        "obat_zip_name": obat_zip_path.name,
        "obat_zip_size": obat_zip_path.stat().st_size,
        "tile_names": sorted(path.name for path in tile_paths),
        "chunk_size": chunk_size,
    }


def load_country_checkpoint(
    checkpoint_path: Path,
    expected_signature: dict,
) -> dict | None:
    if not checkpoint_path.exists():
        return None

    try:
        checkpoint = json.loads(checkpoint_path.read_text(encoding="utf-8"))
    except Exception:
        checkpoint_path.rename(
            checkpoint_path.with_suffix(checkpoint_path.suffix + ".invalid")
        )
        return None

    if checkpoint.get("signature") != expected_signature:
        archived_path = checkpoint_path.with_name(
            checkpoint_path.stem
            + "_incompatible_"
            + datetime.now().strftime("%Y%m%d_%H%M%S")
            + checkpoint_path.suffix
        )
        checkpoint_path.rename(archived_path)
        print("Archived incompatible checkpoint:", archived_path)
        return None

    return checkpoint


def save_country_checkpoint(checkpoint_path: Path, checkpoint: dict) -> None:
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = Path(str(checkpoint_path) + ".part")
    temporary_path.write_text(
        json.dumps(checkpoint, indent=2),
        encoding="utf-8",
    )
    temporary_path.replace(checkpoint_path)


def clear_country_checkpoint(iso3: str, year: int) -> None:
    get_country_checkpoint_path(iso3, year).unlink(missing_ok=True)


def analyse_country_exposure(
    iso3: str,
    year: int,
    obat_zip_path: Path,
    gabam_tile_paths: list[Path],
    chunk_size: int = CHUNK_SIZE,
    max_chunks: int | None = None,
    use_checkpoints: bool = ENABLE_CHECKPOINTS,
) -> pd.DataFrame:
    iso3 = iso3.strip().upper()
    if not gabam_tile_paths:
        raise ValueError(f"No GABAM tiles were supplied for {iso3}.")

    csv_members = find_obat_csv_members(obat_zip_path)
    checkpoint_path = get_country_checkpoint_path(iso3, year)
    signature = build_country_checkpoint_signature(
        iso3,
        year,
        obat_zip_path,
        gabam_tile_paths,
        chunk_size,
    )

    counters = {
        "total_buildings": 0,
        "valid_coordinate_buildings": 0,
        "invalid_coordinate_buildings": 0,
        "tile_covered_buildings": 0,
        "valid_gabam_pixel_buildings": 0,
        "exposed_buildings": 0,
        "processed_chunks": 0,
    }
    resume_part_number = 1
    resume_chunk_in_part = 0

    if use_checkpoints:
        checkpoint = load_country_checkpoint(checkpoint_path, signature)
        if checkpoint is not None:
            counters.update(checkpoint.get("counters", {}))
            resume_part_number = int(checkpoint.get("part_number", 1))
            resume_chunk_in_part = int(
                checkpoint.get("chunk_number_in_part", 0)
            )
            print(
                f"Resuming {iso3} at CSV part {resume_part_number}, "
                f"chunk {resume_chunk_in_part + 1}."
            )

    stop_processing = False
    last_completed_part_number = resume_part_number
    last_completed_chunk_in_part = resume_chunk_in_part

    def write_checkpoint(part_number: int, chunk_number_in_part: int) -> None:
        if not use_checkpoints:
            return
        if counters["processed_chunks"] % CHECKPOINT_EVERY_CHUNKS != 0:
            return
        save_country_checkpoint(
            checkpoint_path,
            {
                "signature": signature,
                "part_number": part_number,
                "chunk_number_in_part": chunk_number_in_part,
                "counters": counters,
                "updated_at_utc": datetime.now(timezone.utc).isoformat(),
            },
        )

    with ExitStack() as stack:
        tile_sources = [
            stack.enter_context(rasterio.open(path))
            for path in gabam_tile_paths
        ]
        archive = stack.enter_context(ZipFile(obat_zip_path, "r"))

        for part_number, csv_member in enumerate(csv_members, start=1):
            if part_number < resume_part_number:
                continue

            print(
                f"Reading CSV part {part_number}/{len(csv_members)}: "
                f"{csv_member}"
            )

            with archive.open(csv_member) as csv_file:
                chunk_reader = pd.read_csv(
                    csv_file,
                    usecols=["lon", "lat"],
                    dtype={"lon": "float64", "lat": "float64"},
                    chunksize=chunk_size,
                    low_memory=False,
                )

                for chunk_number_in_part, chunk in enumerate(
                    chunk_reader,
                    start=1,
                ):
                    if (
                        part_number == resume_part_number
                        and chunk_number_in_part <= resume_chunk_in_part
                    ):
                        continue

                    if (
                        max_chunks is not None
                        and counters["processed_chunks"] >= max_chunks
                    ):
                        stop_processing = True
                        break

                    counters["processed_chunks"] += 1
                    chunk_count = len(chunk)
                    counters["total_buildings"] += chunk_count

                    valid_coordinates = (
                        chunk["lon"].notna()
                        & chunk["lat"].notna()
                        & chunk["lon"].between(-180, 180)
                        & chunk["lat"].between(-90, 90)
                    )
                    valid_count = int(valid_coordinates.sum())
                    counters["valid_coordinate_buildings"] += valid_count
                    counters["invalid_coordinate_buildings"] += (
                        chunk_count - valid_count
                    )

                    valid_chunk = chunk.loc[valid_coordinates]
                    if not valid_chunk.empty:
                        values, tile_covered, valid_pixel = (
                            sample_gabam_values_from_sources(
                                tile_sources,
                                valid_chunk["lon"].to_numpy(),
                                valid_chunk["lat"].to_numpy(),
                            )
                        )
                        exposed = classify_wildfire_exposure(values)

                        counters["tile_covered_buildings"] += int(
                            tile_covered.sum()
                        )
                        counters["valid_gabam_pixel_buildings"] += int(
                            valid_pixel.sum()
                        )
                        counters["exposed_buildings"] += int(exposed.sum())

                    last_completed_part_number = part_number
                    last_completed_chunk_in_part = chunk_number_in_part
                    write_checkpoint(part_number, chunk_number_in_part)

                    print(
                        f"Chunk {counters['processed_chunks']:,}: "
                        f"{chunk_count:,} buildings | "
                        f"cumulative exposed {counters['exposed_buildings']:,}"
                    )

            if stop_processing:
                break

    valid_coordinates = counters["valid_coordinate_buildings"]
    total_buildings = counters["total_buildings"]
    tile_covered = counters["tile_covered_buildings"]
    valid_pixels = counters["valid_gabam_pixel_buildings"]
    exposed = counters["exposed_buildings"]

    result = pd.DataFrame(
        [
            {
                "iso3": iso3,
                "analysis_year": year,
                "gabam_version": GABAM_VERSION,
                "gabam_record_id": GABAM_RECORD_ID,
                "gabam_archive_md5": GABAM_ARCHIVE_MD5,
                "ghs_obat_release": "R2024A",
                "ghs_obat_epoch": "E2020",
                "burned_value_threshold": BURNED_VALUE_THRESHOLD,
                "total_buildings": total_buildings,
                "valid_coordinate_buildings": valid_coordinates,
                "invalid_coordinate_buildings": counters[
                    "invalid_coordinate_buildings"
                ],
                "tile_covered_buildings": tile_covered,
                "tile_uncovered_valid_coordinate_buildings": (
                    valid_coordinates - tile_covered
                ),
                "valid_gabam_pixel_buildings": valid_pixels,
                "invalid_or_nodata_gabam_pixel_buildings": (
                    tile_covered - valid_pixels
                ),
                "exposed_buildings": exposed,
                "exposed_buildings_percent_total": calculate_percentage(
                    exposed,
                    total_buildings,
                ),
                "exposed_buildings_percent_valid_coordinates": (
                    calculate_percentage(exposed, valid_coordinates)
                ),
                "exposed_buildings_percent_valid_gabam_pixels": (
                    calculate_percentage(exposed, valid_pixels)
                ),
                "tile_coverage_percent_valid_coordinates": (
                    calculate_percentage(tile_covered, valid_coordinates)
                ),
                "valid_pixel_coverage_percent_tile_covered": (
                    calculate_percentage(valid_pixels, tile_covered)
                ),
                "gabam_tile_count": len(gabam_tile_paths),
                "obat_csv_part_count": len(csv_members),
                "processed_chunks": counters["processed_chunks"],
                "partial_test": max_chunks is not None,
                "last_completed_part_number": last_completed_part_number,
                "last_completed_chunk_in_part": (
                    last_completed_chunk_in_part
                ),
            }
        ]
    )

    if max_chunks is None and use_checkpoints:
        clear_country_checkpoint(iso3, year)

    return result


### Optional partial test

This cell processes only the first configured number of chunks and is intended to verify ZIP streaming, coordinate validation, raster sampling and checkpoint logic before a full-country run.


In [31]:
test_partial_exposure = None

if TEST_MODE and RUN_PARTIAL_TEST:
    test_partial_exposure = analyse_country_exposure(
        iso3=TEST_COUNTRIES[0],
        year=ANALYSIS_YEAR,
        obat_zip_path=test_obat_zip_path,
        gabam_tile_paths=test_gabam_tile_paths,
        max_chunks=PARTIAL_TEST_MAX_CHUNKS,
        use_checkpoints=False,
    )
    display(test_partial_exposure)


## 19. Run an optional complete single-country test

When enabled, every GHS-OBAT record for the selected test country is processed. The result is not accepted as a completed production output until the validation section succeeds.


In [32]:
full_country_exposure = None
analysis_duration_minutes = np.nan
country_result_path = None

if TEST_MODE and RUN_FULL_COUNTRY_TEST:
    started = perf_counter()
    full_country_exposure = analyse_country_exposure(
        iso3=TEST_COUNTRIES[0],
        year=ANALYSIS_YEAR,
        obat_zip_path=test_obat_zip_path,
        gabam_tile_paths=test_gabam_tile_paths,
        max_chunks=None,
        use_checkpoints=ENABLE_CHECKPOINTS,
    )
    analysis_duration_minutes = (perf_counter() - started) / 60
    display(full_country_exposure)
    print(f"Processing time: {analysis_duration_minutes:.2f} minutes")


## 20. Validate and protect the country result

Validation confirms count identities, ordering constraints, percentage ranges, year/version fields and tile-coverage quality. Output is written through a temporary file so an interrupted write cannot replace a valid result.


In [33]:
COUNTRY_RESULT_COLUMNS = [
    "iso3",
    "analysis_year",
    "gabam_version",
    "gabam_record_id",
    "gabam_archive_md5",
    "ghs_obat_release",
    "ghs_obat_epoch",
    "burned_value_threshold",
    "total_buildings",
    "valid_coordinate_buildings",
    "invalid_coordinate_buildings",
    "tile_covered_buildings",
    "tile_uncovered_valid_coordinate_buildings",
    "valid_gabam_pixel_buildings",
    "invalid_or_nodata_gabam_pixel_buildings",
    "exposed_buildings",
    "exposed_buildings_percent_total",
    "exposed_buildings_percent_valid_coordinates",
    "exposed_buildings_percent_valid_gabam_pixels",
    "tile_coverage_percent_valid_coordinates",
    "valid_pixel_coverage_percent_tile_covered",
    "gabam_tile_count",
    "obat_csv_part_count",
    "processed_chunks",
    "partial_test",
    "last_completed_part_number",
    "last_completed_chunk_in_part",
]


def get_country_output_path(iso3: str, year: int) -> Path:
    return get_country_output_directory(year) / f"{iso3}_GABAM{year}.csv"


def validate_country_result(
    result: pd.DataFrame,
    output_path: Path | None = None,
) -> None:
    if len(result) != 1:
        raise ValueError("Country result must contain exactly one row.")

    missing = set(COUNTRY_RESULT_COLUMNS) - set(result.columns)
    if missing:
        raise ValueError(
            f"Country result is missing columns: {sorted(missing)}"
        )

    row = result.iloc[0]
    if int(row["analysis_year"]) != ANALYSIS_YEAR:
        raise ValueError("Country result analysis year is inconsistent.")

    total = int(row["total_buildings"])
    valid = int(row["valid_coordinate_buildings"])
    invalid = int(row["invalid_coordinate_buildings"])
    tile_covered = int(row["tile_covered_buildings"])
    tile_uncovered = int(row["tile_uncovered_valid_coordinate_buildings"])
    valid_pixels = int(row["valid_gabam_pixel_buildings"])
    nodata = int(row["invalid_or_nodata_gabam_pixel_buildings"])
    exposed = int(row["exposed_buildings"])

    if total != valid + invalid:
        raise ValueError("total_buildings != valid + invalid coordinates")
    if valid != tile_covered + tile_uncovered:
        raise ValueError("valid coordinates != tile covered + uncovered")
    if tile_covered != valid_pixels + nodata:
        raise ValueError("tile covered != valid pixels + NoData pixels")
    if not (0 <= exposed <= valid_pixels <= tile_covered <= valid <= total):
        raise ValueError("Country result count ordering is invalid.")

    percentage_columns = [
        column
        for column in result.columns
        if "percent" in column
    ]
    for column in percentage_columns:
        value = row[column]
        if pd.notna(value) and not 0 <= float(value) <= 100:
            raise ValueError(f"{column} is outside 0–100: {value}")

    coverage = row["tile_coverage_percent_valid_coordinates"]
    if (
        pd.notna(coverage)
        and float(coverage) < MIN_TILE_COVERAGE_PERCENT
        and FAIL_ON_LOW_TILE_COVERAGE
    ):
        raise ValueError(
            f"Tile coverage {coverage:.3f}% is below the configured "
            f"minimum {MIN_TILE_COVERAGE_PERCENT:.3f}%."
        )

    if output_path is not None:
        output_path.parent.mkdir(parents=True, exist_ok=True)
        temporary_path = Path(str(output_path) + ".part")
        result[COUNTRY_RESULT_COLUMNS].to_csv(temporary_path, index=False)
        reloaded = pd.read_csv(temporary_path)
        if len(reloaded) != 1:
            temporary_path.unlink(missing_ok=True)
            raise IOError("Temporary country CSV could not be reloaded.")
        temporary_path.replace(output_path)


In [34]:
if TEST_MODE and RUN_FULL_COUNTRY_TEST:
    country_result_path = get_country_output_path(
        TEST_COUNTRIES[0],
        ANALYSIS_YEAR,
    )
    validate_country_result(
        full_country_exposure,
        country_result_path,
    )
    print("Validated country output:", country_result_path)


## 21. Record outcomes and quarantine invalid files

Invalid existing country CSVs are moved to a quarantine directory rather than silently overwritten. Every country attempt appends a structured row to the year-specific processing log.


In [35]:
PROCESSING_LOG_COLUMNS = [
    "timestamp_utc",
    "iso3",
    "analysis_year",
    "status",
    "stage",
    "started_at_utc",
    "finished_at_utc",
    "processing_minutes",
    "output_path",
    "total_buildings",
    "valid_coordinate_buildings",
    "tile_covered_buildings",
    "valid_gabam_pixel_buildings",
    "exposed_buildings",
    "exposed_buildings_percent_total",
    "tile_coverage_percent_valid_coordinates",
    "message",
]


def current_utc_time() -> str:
    return datetime.now(timezone.utc).isoformat()


def initialise_processing_log(year: int = ANALYSIS_YEAR) -> Path:
    path = get_processing_log_path(year)
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        pd.DataFrame(columns=PROCESSING_LOG_COLUMNS).to_csv(path, index=False)
    return path


def append_processing_log(
    iso3: str,
    year: int,
    status: str,
    stage: str,
    started_at_utc: str,
    finished_at_utc: str,
    processing_minutes: float,
    output_path: Path | None = None,
    result: pd.DataFrame | None = None,
    message: str = "",
) -> None:
    path = initialise_processing_log(year)
    result_row = result.iloc[0] if result is not None and not result.empty else {}

    record = {
        "timestamp_utc": current_utc_time(),
        "iso3": iso3,
        "analysis_year": year,
        "status": status,
        "stage": stage,
        "started_at_utc": started_at_utc,
        "finished_at_utc": finished_at_utc,
        "processing_minutes": processing_minutes,
        "output_path": str(output_path) if output_path else "",
        "total_buildings": result_row.get("total_buildings", np.nan),
        "valid_coordinate_buildings": result_row.get(
            "valid_coordinate_buildings", np.nan
        ),
        "tile_covered_buildings": result_row.get(
            "tile_covered_buildings", np.nan
        ),
        "valid_gabam_pixel_buildings": result_row.get(
            "valid_gabam_pixel_buildings", np.nan
        ),
        "exposed_buildings": result_row.get("exposed_buildings", np.nan),
        "exposed_buildings_percent_total": result_row.get(
            "exposed_buildings_percent_total", np.nan
        ),
        "tile_coverage_percent_valid_coordinates": result_row.get(
            "tile_coverage_percent_valid_coordinates", np.nan
        ),
        "message": message,
    }

    pd.DataFrame([record], columns=PROCESSING_LOG_COLUMNS).to_csv(
        path,
        mode="a",
        header=False,
        index=False,
    )


def quarantine_invalid_country_output(
    output_path: Path,
    year: int,
) -> Path | None:
    if not output_path.exists():
        return None

    destination = get_invalid_output_directory(year) / (
        output_path.stem
        + "_invalid_"
        + datetime.now().strftime("%Y%m%d_%H%M%S")
        + output_path.suffix
    )
    destination.parent.mkdir(parents=True, exist_ok=True)
    output_path.replace(destination)
    return destination


initialise_processing_log()
display(pd.read_csv(PROCESSING_LOG_PATH).tail())


,timestamp_utc,iso3,analysis_year,status,stage,started_at_utc,finished_at_utc,processing_minutes,output_path,total_buildings,valid_coordinate_buildings,tile_covered_buildings,valid_gabam_pixel_buildings,exposed_buildings,exposed_buildings_percent_total,tile_coverage_percent_valid_coordinates,message
266,2026-08-04T04:41:30.344254+00:00,REU,2024,skipped,select_gabam_tiles,2026-08-04T04:41:30.328881+00:00,2026-08-04T04:41:30.342184+00:00,0.000222,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No intersecting GABAM tiles were found.
267,2026-08-04T04:42:11.927078+00:00,SHN,2024,skipped,select_gabam_tiles,2026-08-04T04:42:11.920237+00:00,2026-08-04T04:42:11.925079+00:00,0.000081,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No intersecting GABAM tiles were found.
268,2026-08-04T04:42:45.784450+00:00,SPM,2024,skipped,select_gabam_tiles,2026-08-04T04:42:45.770033+00:00,2026-08-04T04:42:45.782475+00:00,0.000207,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No intersecting GABAM tiles were found.
269,2026-08-04T04:45:00.515207+00:00,SGS,2024,skipped,select_gabam_tiles,2026-08-04T04:45:00.493877+00:00,2026-08-04T04:45:00.513046+00:00,0.000320,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No intersecting GABAM tiles were found.
270,2026-08-04T04:45:26.685567+00:00,LKA,2024,skipped,select_gabam_tiles,2026-08-04T04:45:26.646373+00:00,2026-08-04T04:45:26.682757+00:00,0.000607,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No intersecting GABAM tiles were found.


## 22. Prepare country-processing and cleanup helpers

These helpers reuse completed outputs, resolve the global GABAM archive/catalogue, remove country-specific raw files when requested, and define expected skip conditions.


In [36]:
class SkipCountryError(Exception):
    """Raised for an expected country-level skip."""


def load_completed_country_result(
    iso3: str,
    year: int,
) -> pd.DataFrame | None:
    output_path = get_country_output_path(iso3, year)
    if not output_path.exists():
        return None

    try:
        result = pd.read_csv(output_path)
        validate_country_result(result)
        return result
    except Exception as error:
        quarantined = quarantine_invalid_country_output(output_path, year)
        print(
            f"Quarantined invalid existing output for {iso3}: "
            f"{quarantined}; reason: {error}"
        )
        return None


def remove_country_raw_files(iso3: str) -> None:
    if KEEP_BUILDING_RAW_DATA:
        return
    path = get_ghs_obat_zip_path(iso3)
    path.unlink(missing_ok=True)
    if path.parent.exists() and not any(path.parent.iterdir()):
        path.parent.rmdir()


def remove_gabam_tile_cache(tile_paths: list[Path]) -> None:
    if KEEP_GABAM_TILE_CACHE:
        return
    for path in tile_paths:
        path.unlink(missing_ok=True)


def remove_gabam_archive() -> None:
    if KEEP_GABAM_ARCHIVE:
        return
    get_gabam_archive_path().unlink(missing_ok=True)
    GABAM_CHECKSUM_MARKER_PATH.unlink(missing_ok=True)


_GABAM_ARCHIVE_PATH_CACHE: Path | None = None
_GABAM_TILE_CATALOGUE_CACHE: gpd.GeoDataFrame | None = None


def ensure_gabam_resources() -> tuple[Path, gpd.GeoDataFrame]:
    global _GABAM_ARCHIVE_PATH_CACHE
    global _GABAM_TILE_CATALOGUE_CACHE

    if (
        _GABAM_ARCHIVE_PATH_CACHE is not None
        and _GABAM_ARCHIVE_PATH_CACHE.exists()
        and _GABAM_TILE_CATALOGUE_CACHE is not None
    ):
        return (
            _GABAM_ARCHIVE_PATH_CACHE,
            _GABAM_TILE_CATALOGUE_CACHE,
        )

    archive_record = resolve_gabam_archive()
    archive_path = download_gabam_archive(archive_record)
    catalogue = build_gabam_tile_catalogue(archive_path)
    _GABAM_ARCHIVE_PATH_CACHE = archive_path
    _GABAM_TILE_CATALOGUE_CACHE = catalogue
    return archive_path, catalogue


## 23. Process one country from start to finish

`process_country()` is the formal end-to-end processing unit. It manages source resolution, tile selection/extraction, value-encoding checks, GHS-OBAT streaming, validation, logging, cleanup and expected skips.


In [37]:
def process_country(
    iso3: str,
    year: int = ANALYSIS_YEAR,
) -> pd.DataFrame | None:
    iso3 = iso3.strip().upper()
    started_at_utc = current_utc_time()
    started = perf_counter()
    stage = "initialise"
    output_path = get_country_output_path(iso3, year)
    tile_paths: list[Path] = []
    result = None

    try:
        if SKIP_COMPLETED:
            completed = load_completed_country_result(iso3, year)
            if completed is not None:
                print(f"Reusing completed output for {iso3}.")
                return completed

        stage = "resolve_boundary"
        boundary = get_country_boundary(iso3)
        if boundary.geometry.iloc[0].intersection(
            GABAM_COVERAGE_GEOMETRY
        ).is_empty:
            raise SkipCountryError(
                "Country boundary is outside GABAM's published "
                "80°N to 60°S latitude coverage."
            )

        stage = "resolve_gabam_archive"
        archive_path, tile_catalogue = ensure_gabam_resources()

        stage = "select_gabam_tiles"
        tile_table = find_intersecting_gabam_tiles(iso3, tile_catalogue)
        if tile_table.empty:
            raise SkipCountryError("No intersecting GABAM tiles were found.")

        stage = "extract_gabam_tiles"
        tile_paths = extract_gabam_tiles(archive_path, tile_table)
        inspect_gabam_tiles(tile_paths)

        stage = "validate_gabam_encoding"
        inspect_gabam_value_encoding(tile_paths)

        stage = "resolve_ghs_obat"
        obat_record = create_ghs_obat_download_record(iso3)
        if not obat_record["accessible"]:
            raise SkipCountryError(
                f"GHS-OBAT country file is unavailable: {obat_record['error']}"
            )

        stage = "download_ghs_obat"
        obat_zip_path = download_ghs_obat_zip(iso3, obat_record)
        inspect_ghs_obat_zip(obat_zip_path, sample_rows=2)

        stage = "analyse_buildings"
        result = analyse_country_exposure(
            iso3=iso3,
            year=year,
            obat_zip_path=obat_zip_path,
            gabam_tile_paths=tile_paths,
            chunk_size=CHUNK_SIZE,
            max_chunks=None,
            use_checkpoints=ENABLE_CHECKPOINTS,
        )

        coverage = result.iloc[0][
            "tile_coverage_percent_valid_coordinates"
        ]
        status = "completed"
        message = ""

        if (
            pd.notna(coverage)
            and float(coverage) < MIN_TILE_COVERAGE_PERCENT
        ):
            status = "completed_low_coverage"
            message = (
                f"Tile coverage {coverage:.3f}% is below "
                f"{MIN_TILE_COVERAGE_PERCENT:.3f}%."
            )

        stage = "validate_output"
        validate_country_result(result, output_path)

        finished_at_utc = current_utc_time()
        minutes = (perf_counter() - started) / 60
        append_processing_log(
            iso3=iso3,
            year=year,
            status=status,
            stage="completed",
            started_at_utc=started_at_utc,
            finished_at_utc=finished_at_utc,
            processing_minutes=minutes,
            output_path=output_path,
            result=result,
            message=message,
        )

        remove_country_raw_files(iso3)
        remove_gabam_tile_cache(tile_paths)
        gc.collect()
        return result

    except SkipCountryError as error:
        finished_at_utc = current_utc_time()
        minutes = (perf_counter() - started) / 60
        append_processing_log(
            iso3=iso3,
            year=year,
            status="skipped",
            stage=stage,
            started_at_utc=started_at_utc,
            finished_at_utc=finished_at_utc,
            processing_minutes=minutes,
            output_path=None,
            result=result,
            message=str(error),
        )
        clear_country_checkpoint(iso3, year)
        remove_country_raw_files(iso3)
        remove_gabam_tile_cache(tile_paths)
        print(f"Skipped {iso3}: {error}")
        return None

    except Exception as error:
        finished_at_utc = current_utc_time()
        minutes = (perf_counter() - started) / 60
        append_processing_log(
            iso3=iso3,
            year=year,
            status="failed",
            stage=stage,
            started_at_utc=started_at_utc,
            finished_at_utc=finished_at_utc,
            processing_minutes=minutes,
            output_path=output_path if output_path.exists() else None,
            result=result,
            message=f"{type(error).__name__}: {error}",
        )
        print(f"Failed {iso3} during {stage}: {error}")
        traceback.print_exc()
        return None


In [38]:
test_process_country_result = None

if TEST_MODE and RUN_PROCESS_COUNTRY_TEST:
    test_process_country_result = process_country(
        TEST_COUNTRIES[0],
        ANALYSIS_YEAR,
    )
    if test_process_country_result is not None:
        display(test_process_country_result)


## 24. Run a selected multi-country sequence

Country runs remain independent. A failure or expected skip does not stop later countries, and valid completed outputs can be reused.


In [39]:
def run_country_loop(
    country_codes: list[str],
    year: int = ANALYSIS_YEAR,
) -> pd.DataFrame:
    results = []
    for position, iso3 in enumerate(country_codes, start=1):
        print("=" * 80)
        print(f"Country {position}/{len(country_codes)}: {iso3}")
        country_result = process_country(iso3, year)
        if country_result is not None:
            results.append(country_result)

    if not results:
        return pd.DataFrame(columns=COUNTRY_RESULT_COLUMNS)

    return pd.concat(results, ignore_index=True)


SMALL_TEST_COUNTRIES = ["LUX", "BRN", "SGP"]
small_loop_results = pd.DataFrame()

if TEST_MODE and RUN_SMALL_MULTI_COUNTRY_TEST:
    small_loop_results = run_country_loop(
        SMALL_TEST_COUNTRIES,
        ANALYSIS_YEAR,
    )
    display(small_loop_results)


## 25. Prepare test or global execution

The global runner is protected by `TEST_MODE`. Logs are not reset unless explicitly requested. Country codes are validated against `ALL_COUNTRIES`, and global output directories are created before processing begins.


In [40]:
RESET_PROCESSING_LOG = False

if RESET_PROCESSING_LOG:
    PROCESSING_LOG_PATH.unlink(missing_ok=True)
    initialise_processing_log()

if TEST_MODE:
    RUN_COUNTRIES = TEST_COUNTRIES.copy()
else:
    RUN_COUNTRIES = ALL_COUNTRIES.copy()

invalid_run_codes = [
    iso3 for iso3 in RUN_COUNTRIES if iso3 not in ALL_COUNTRIES
]
if invalid_run_codes:
    raise ValueError(f"Invalid run country codes: {invalid_run_codes}")

for directory in [
    get_country_output_directory(ANALYSIS_YEAR),
    get_global_map_output_directory(ANALYSIS_YEAR),
    get_invalid_output_directory(ANALYSIS_YEAR),
    get_checkpoint_directory(ANALYSIS_YEAR),
    get_log_directory(ANALYSIS_YEAR),
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Run mode:", "test" if TEST_MODE else "global")
print("Countries selected:", len(RUN_COUNTRIES))
print("First codes:", RUN_COUNTRIES[:10])


Run mode: test
Countries selected: 1
First codes: ['LUX']


## 26. Run all countries in automatic batches

The country list is divided into configurable sequential batches. A cumulative global progress CSV is rebuilt from all valid country outputs after each batch.


In [41]:
def update_global_progress_file(year: int) -> Path | None:
    country_directory = get_country_output_directory(year)
    valid_results = []

    for output_path in sorted(country_directory.glob(f"*_GABAM{year}.csv")):
        try:
            result = pd.read_csv(output_path)
            validate_country_result(result)
            valid_results.append(result[COUNTRY_RESULT_COLUMNS])
        except Exception as error:
            print(f"Ignoring invalid country output {output_path.name}: {error}")

    if not valid_results:
        return None

    global_progress = (
        pd.concat(valid_results, ignore_index=True)
        .drop_duplicates(subset=["iso3"], keep="last")
        .sort_values("iso3")
        .reset_index(drop=True)
    )

    output_path = get_output_directory(year) / f"global_progress_GABAM{year}.csv"
    temporary_path = Path(str(output_path) + ".part")
    global_progress.to_csv(temporary_path, index=False)
    temporary_path.replace(output_path)
    return output_path


def run_all_country_batches(
    country_codes: list[str],
    year: int = ANALYSIS_YEAR,
    batch_size: int = 5,
    delay_between_batches: int = 5,
) -> Path | None:
    if batch_size < 1:
        raise ValueError("batch_size must be at least 1.")

    total_batches = ceil(len(country_codes) / batch_size)
    progress_path = None

    for batch_number in range(1, total_batches + 1):
        start_index = (batch_number - 1) * batch_size
        batch_codes = country_codes[start_index : start_index + batch_size]

        print("#" * 80)
        print(
            f"Batch {batch_number}/{total_batches}: "
            f"{', '.join(batch_codes)}"
        )
        run_country_loop(batch_codes, year)
        progress_path = update_global_progress_file(year)
        print("Global progress file:", progress_path)

        if batch_number < total_batches and delay_between_batches > 0:
            sleep(delay_between_batches)

    if GENERATE_GLOBAL_MAPS:
        if progress_path is not None and Path(progress_path).exists():
            create_global_map_outputs(year)
        else:
            print(
                "Global-map generation skipped: no validated country "
                "results are currently available."
            )
            print(
                "Review the latest rows in:",
                get_processing_log_path(year),
            )

    remove_gabam_archive()
    return progress_path


In [42]:
print("Analysis year:", ANALYSIS_YEAR)
print("Countries to process:", len(RUN_COUNTRIES))
print("Default global batch size: 5")


Analysis year: 2024
Countries to process: 1
Default global batch size: 5


## 27. Generate global exposure maps

Global map products use validated country CSVs and the prepared country boundaries. The default mapped field is `exposed_buildings_percent_total`; data-coverage fields remain available in the CSV and GeoPackage for interpretation.


In [43]:
GLOBAL_COUNTRY_LAYER_NAME = "wildfire_building_exposure"


def get_global_progress_path(year: int) -> Path:
    return get_output_directory(year) / f"global_progress_GABAM{year}.csv"


def get_global_map_output_paths(year: int) -> dict[str, Path]:
    directory = get_global_map_output_directory(year)
    return {
        "csv": get_global_progress_path(year),
        "gpkg": directory / f"global_wildfire_exposure_GABAM{year}.gpkg",
        "html": directory / f"global_wildfire_exposure_GABAM{year}.html",
        "png": directory / f"global_wildfire_exposure_GABAM{year}.png",
    }


def prepare_global_exposure_geodataframe(year: int) -> gpd.GeoDataFrame:
    progress_path = get_global_progress_path(year)
    if not progress_path.exists():
        updated = update_global_progress_file(year)
        if updated is None:
            raise FileNotFoundError(
                "No validated country results are available for global maps."
            )

    results = pd.read_csv(progress_path)
    merged = country_boundaries.merge(results, on="iso3", how="left")
    return gpd.GeoDataFrame(merged, geometry="geometry", crs="EPSG:4326")


def create_global_static_map(
    exposure_gdf: gpd.GeoDataFrame,
    output_path: Path,
    year: int,
) -> Path:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    figure, axis = plt.subplots(figsize=(18, 10))

    exposure_gdf.plot(
        column="exposed_buildings_percent_total",
        ax=axis,
        legend=True,
        missing_kwds={"label": "No validated result"},
        edgecolor="0.45",
        linewidth=0.25,
    )
    axis.set_title(
        f"Indicator 6.1.1 — Buildings exposed to GABAM burned area ({year})"
    )
    axis.set_axis_off()
    figure.tight_layout()
    figure.savefig(output_path, dpi=220, bbox_inches="tight")
    plt.close(figure)
    return output_path


def create_global_interactive_map(
    exposure_gdf: gpd.GeoDataFrame,
    output_path: Path,
    year: int,
) -> Path:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    map_object = folium.Map(
        location=[15, 0],
        zoom_start=2,
        tiles=GLOBAL_MAP_BASE_TILES,
    )

    valid_values = exposure_gdf[
        "exposed_buildings_percent_total"
    ].dropna()
    maximum = max(float(valid_values.max()) if not valid_values.empty else 1, 1)
    colormap = linear.YlOrRd_09.scale(0, maximum)
    colormap.caption = (
        f"Buildings exposed to GABAM burned area (% of total), {year}"
    )
    colormap.add_to(map_object)

    def style_function(feature):
        value = feature["properties"].get(
            "exposed_buildings_percent_total"
        )
        if value is None:
            return {
                "fillColor": "#d9d9d9",
                "color": "#777777",
                "weight": 0.4,
                "fillOpacity": 0.4,
            }
        return {
            "fillColor": colormap(float(value)),
            "color": "#555555",
            "weight": 0.4,
            "fillOpacity": 0.75,
        }

    tooltip_fields = [
        "iso3",
        "country_name",
        "exposed_buildings",
        "total_buildings",
        "exposed_buildings_percent_total",
        "tile_coverage_percent_valid_coordinates",
    ]
    tooltip_aliases = [
        "ISO3",
        "Country / territory",
        "Exposed buildings",
        "Total buildings",
        "Exposure (% total)",
        "Tile coverage (% valid coordinates)",
    ]

    folium.GeoJson(
        exposure_gdf.to_json(),
        name="Wildfire building exposure",
        style_function=style_function,
        tooltip=folium.GeoJsonTooltip(
            fields=tooltip_fields,
            aliases=tooltip_aliases,
            localize=True,
        ),
    ).add_to(map_object)

    folium.LayerControl().add_to(map_object)
    map_object.save(output_path)
    return output_path


def create_global_map_outputs(year: int = ANALYSIS_YEAR) -> dict[str, Path]:
    exposure_gdf = prepare_global_exposure_geodataframe(year)
    paths = get_global_map_output_paths(year)
    outputs = {"csv": paths["csv"]}

    if SAVE_GLOBAL_GPKG:
        paths["gpkg"].unlink(missing_ok=True)
        exposure_gdf.to_file(
            paths["gpkg"],
            layer=GLOBAL_COUNTRY_LAYER_NAME,
            driver="GPKG",
        )
        outputs["gpkg"] = paths["gpkg"]

    if GENERATE_GLOBAL_INTERACTIVE_HTML:
        outputs["html"] = create_global_interactive_map(
            exposure_gdf,
            paths["html"],
            year,
        )

    if GENERATE_GLOBAL_STATIC_PNG:
        outputs["png"] = create_global_static_map(
            exposure_gdf,
            paths["png"],
            year,
        )

    print("Global map outputs:")
    for name, path in outputs.items():
        print(f"- {name}: {path}")
    return outputs


### Optional map-only regeneration

This switch rebuilds global products from existing validated country CSVs without reading building records again.


In [44]:
RUN_GLOBAL_MAP_ONLY = False

global_map_only_outputs = None
if RUN_GLOBAL_MAP_ONLY:
    global_map_only_outputs = create_global_map_outputs(ANALYSIS_YEAR)


## 28. Monitor live global-processing progress

The live display reports completed outputs, latest log status, elapsed time and current country/batch while the automated runner is active.


In [45]:
def format_elapsed_time(elapsed_seconds: float) -> str:
    elapsed_seconds = max(0, int(elapsed_seconds))
    hours, remainder = divmod(elapsed_seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"


def get_completed_output_codes(year: int) -> set[str]:
    codes = set()
    for path in get_country_output_directory(year).glob(f"*_GABAM{year}.csv"):
        result = load_completed_country_result(path.name[:3], year)
        if result is not None:
            codes.add(str(result.iloc[0]["iso3"]))
    return codes


def get_latest_log_summary(year: int) -> dict:
    path = get_processing_log_path(year)
    if not path.exists() or path.stat().st_size == 0:
        return {}
    log = pd.read_csv(path)
    if log.empty:
        return {}
    return log.iloc[-1].to_dict()


def create_global_progress_display(
    current_position: int,
    total_countries: int,
    current_iso3: str,
    current_status: str,
    batch_number: int,
    total_batches: int,
    elapsed_seconds: float,
    year: int,
) -> HTML:
    completed_count = len(get_completed_output_codes(year))
    percent = (
        100 * current_position / total_countries
        if total_countries
        else 0
    )
    latest = get_latest_log_summary(year)

    content = f"""
    <div style='border:1px solid #9aa; border-radius:8px; padding:12px;'>
      <h3 style='margin-top:0;'>Indicator 6.1.1 processing progress</h3>
      <b>GABAM year:</b> {year}<br>
      <b>Country:</b> {html.escape(current_iso3)}<br>
      <b>Status:</b> {html.escape(current_status)}<br>
      <b>Position:</b> {current_position}/{total_countries} ({percent:.1f}%)<br>
      <b>Batch:</b> {batch_number}/{total_batches}<br>
      <b>Validated country outputs:</b> {completed_count}<br>
      <b>Elapsed:</b> {format_elapsed_time(elapsed_seconds)}<br>
      <b>Latest log status:</b> {html.escape(str(latest.get('status', '')))}
    </div>
    """
    return HTML(content)


In [46]:
def show_latest_country_attempt(
    year: int,
    iso3: str | None = None,
) -> pd.DataFrame:
    """Display the latest processing-log row for one country or the run."""
    log_path = get_processing_log_path(year)
    if not log_path.exists() or log_path.stat().st_size == 0:
        print("No processing log exists yet:", log_path)
        return pd.DataFrame()

    log = pd.read_csv(log_path)
    if iso3 is not None and "iso3" in log.columns:
        selected = log[
            log["iso3"].astype("string").str.upper() == iso3.upper()
        ]
        if not selected.empty:
            log = selected

    latest = log.tail(1).copy()
    preferred_columns = [
        "iso3",
        "analysis_year",
        "status",
        "stage",
        "message",
        "processing_minutes",
        "output_path",
        "started_at_utc",
        "finished_at_utc",
    ]
    display_columns = [
        column for column in preferred_columns if column in latest.columns
    ]
    display(latest[display_columns])
    return latest


def run_all_country_batches_with_live_display(
    country_codes: list[str],
    year: int = ANALYSIS_YEAR,
    batch_size: int = 5,
    delay_between_batches: int = 5,
) -> Path | None:
    from IPython.display import clear_output

    if batch_size < 1:
        raise ValueError("batch_size must be at least 1.")

    started = perf_counter()
    total_countries = len(country_codes)
    total_batches = ceil(total_countries / batch_size)
    progress_path = None
    unsuccessful_codes = []

    for position, iso3 in enumerate(country_codes, start=1):
        batch_number = ceil(position / batch_size)
        clear_output(wait=True)
        display(
            create_global_progress_display(
                position,
                total_countries,
                iso3,
                "processing",
                batch_number,
                total_batches,
                perf_counter() - started,
                year,
            )
        )

        country_result = process_country(iso3, year)
        progress_path = update_global_progress_file(year)
        latest_log = get_latest_log_summary(year)
        latest_status = str(latest_log.get("status", "unknown"))
        latest_stage = str(latest_log.get("stage", ""))
        latest_message = str(latest_log.get("message", ""))

        if country_result is None:
            unsuccessful_codes.append(iso3)
            display_status = f"{latest_status} during {latest_stage}"
        else:
            display_status = latest_status or "completed"

        clear_output(wait=True)
        display(
            create_global_progress_display(
                position,
                total_countries,
                iso3,
                display_status,
                batch_number,
                total_batches,
                perf_counter() - started,
                year,
            )
        )

        if country_result is None:
            print(f"No validated country output was created for {iso3}.")
            if latest_stage:
                print("Failure/skip stage:", latest_stage)
            if latest_message and latest_message.lower() != "nan":
                print("Reason:", latest_message)
            print("Latest processing-log row:")
            show_latest_country_attempt(year, iso3)
        else:
            print("Validated country output:", get_country_output_path(iso3, year))

        if (
            position < total_countries
            and position % batch_size == 0
            and delay_between_batches > 0
        ):
            sleep(delay_between_batches)

    if GENERATE_GLOBAL_MAPS:
        if progress_path is not None and Path(progress_path).exists():
            create_global_map_outputs(year)
        else:
            print(
                "Global-map generation skipped because the run produced "
                "no validated country result."
            )
            print("Processing log:", get_processing_log_path(year))

    if unsuccessful_codes:
        print(
            "Countries without a validated output:",
            ", ".join(unsuccessful_codes),
        )

    remove_gabam_archive()
    return progress_path


## 29. Interactive workflow dashboard

The dashboard controls the same tested functions used elsewhere in the notebook. It does not start background work: processing runs in the active Jupyter kernel after **Start processing** is pressed.


In [47]:
def create_workflow_dashboard():
    import ipywidgets as widgets

    mode_widget = widgets.ToggleButtons(
        options=[("Test", "test"), ("Global", "global")],
        value="test" if TEST_MODE else "global",
        description="Mode:",
    )
    year_widget = widgets.Dropdown(
        options=AVAILABLE_GABAM_YEARS,
        value=ANALYSIS_YEAR,
        description="GABAM year:",
    )
    test_countries_widget = widgets.Text(
        value=", ".join(TEST_COUNTRIES),
        description="Test ISO3:",
        placeholder="LUX, BRN, SGP",
    )
    batch_size_widget = widgets.BoundedIntText(
        value=5,
        min=1,
        max=100,
        description="Batch size:",
    )
    batch_delay_widget = widgets.BoundedIntText(
        value=5,
        min=0,
        max=3600,
        description="Batch delay:",
    )
    chunk_size_widget = widgets.BoundedIntText(
        value=CHUNK_SIZE,
        min=1,
        max=5_000_000,
        step=10_000,
        description="Chunk size:",
    )
    minimum_coverage_widget = widgets.BoundedFloatText(
        value=MIN_TILE_COVERAGE_PERCENT,
        min=0,
        max=100,
        step=0.1,
        description="Min coverage %:",
    )
    skip_completed_widget = widgets.Checkbox(
        value=SKIP_COMPLETED,
        description="Reuse valid completed outputs",
        indent=False,
    )
    checkpoints_widget = widgets.Checkbox(
        value=ENABLE_CHECKPOINTS,
        description="Enable checkpoints",
        indent=False,
    )
    keep_archive_widget = widgets.Checkbox(
        value=KEEP_GABAM_ARCHIVE,
        description="Keep GABAM global ZIP",
        indent=False,
    )
    keep_tiles_widget = widgets.Checkbox(
        value=KEEP_GABAM_TILE_CACHE,
        description="Keep extracted GABAM tiles",
        indent=False,
    )
    keep_building_widget = widgets.Checkbox(
        value=KEEP_BUILDING_RAW_DATA,
        description="Keep country GHS-OBAT ZIPs",
        indent=False,
    )
    generate_maps_widget = widgets.Checkbox(
        value=GENERATE_GLOBAL_MAPS,
        description="Generate global map products",
        indent=False,
    )
    global_confirmation_widget = widgets.Checkbox(
        value=False,
        description="I confirm complete global processing",
        indent=False,
        layout=widgets.Layout(display="none"),
    )

    validate_button = widgets.Button(
        description="Validate configuration",
        icon="check",
        button_style="info",
    )
    start_button = widgets.Button(
        description="Start processing",
        icon="play",
        button_style="success",
    )
    map_only_button = widgets.Button(
        description="Generate/update global map",
        icon="globe",
        button_style="warning",
    )
    status_widget = widgets.HTML()
    output_widget = widgets.Output(
        layout=widgets.Layout(
            border="1px solid #cccccc",
            padding="10px",
            max_height="720px",
            overflow_y="auto",
        )
    )

    def parse_test_codes() -> list[str]:
        raw_values = (
            test_countries_widget.value
            .replace(";", ",")
            .replace("\n", ",")
            .split(",")
        )
        codes = []
        for raw in raw_values:
            code_value = raw.strip().upper()
            if code_value and code_value not in codes:
                codes.append(code_value)
        return codes

    def read_configuration() -> dict:
        mode = mode_widget.value
        year = int(year_widget.value)
        if year not in AVAILABLE_GABAM_YEARS:
            raise ValueError(f"Unsupported GABAM year: {year}")

        if mode == "test":
            country_codes = parse_test_codes()
            if not country_codes:
                raise ValueError("Enter at least one test ISO3 code.")
            invalid = [code for code in country_codes if code not in ALL_COUNTRIES]
            if invalid:
                raise ValueError(
                    f"Codes not present in countries.csv: {invalid}"
                )
        else:
            if not global_confirmation_widget.value:
                raise ValueError("Global mode requires confirmation.")
            country_codes = ALL_COUNTRIES.copy()

        return {
            "mode": mode,
            "year": year,
            "country_codes": country_codes,
            "batch_size": int(batch_size_widget.value),
            "batch_delay": int(batch_delay_widget.value),
            "chunk_size": int(chunk_size_widget.value),
            "minimum_coverage": float(minimum_coverage_widget.value),
            "skip_completed": bool(skip_completed_widget.value),
            "checkpoints": bool(checkpoints_widget.value),
            "keep_archive": bool(keep_archive_widget.value),
            "keep_tiles": bool(keep_tiles_widget.value),
            "keep_building": bool(keep_building_widget.value),
            "generate_maps": bool(generate_maps_widget.value),
        }

    def apply_configuration(configuration: dict) -> None:
        global ANALYSIS_YEAR
        global CHUNK_SIZE
        global MIN_TILE_COVERAGE_PERCENT
        global SKIP_COMPLETED
        global ENABLE_CHECKPOINTS
        global KEEP_GABAM_ARCHIVE
        global KEEP_GABAM_TILE_CACHE
        global KEEP_BUILDING_RAW_DATA
        global GENERATE_GLOBAL_MAPS

        ANALYSIS_YEAR = configuration["year"]
        CHUNK_SIZE = configuration["chunk_size"]
        MIN_TILE_COVERAGE_PERCENT = configuration["minimum_coverage"]
        SKIP_COMPLETED = configuration["skip_completed"]
        ENABLE_CHECKPOINTS = configuration["checkpoints"]
        KEEP_GABAM_ARCHIVE = configuration["keep_archive"]
        KEEP_GABAM_TILE_CACHE = configuration["keep_tiles"]
        KEEP_BUILDING_RAW_DATA = configuration["keep_building"]
        GENERATE_GLOBAL_MAPS = configuration["generate_maps"]

    def update_mode_visibility(*_args) -> None:
        is_global = mode_widget.value == "global"
        test_countries_widget.layout.display = "none" if is_global else ""
        global_confirmation_widget.layout.display = "" if is_global else "none"
        if not is_global:
            global_confirmation_widget.value = False

    def validate_configuration(_button=None) -> None:
        try:
            configuration = read_configuration()
            status_widget.value = (
                "<div style='border:1px solid #7a9; padding:8px;'>"
                "<b>Configuration valid</b><br>"
                f"Mode: {configuration['mode']}<br>"
                f"Year: {configuration['year']}<br>"
                f"Countries: {len(configuration['country_codes'])}<br>"
                f"Chunk size: {configuration['chunk_size']:,}"
                "</div>"
            )
        except Exception as error:
            status_widget.value = (
                "<div style='border:1px solid #c66; padding:8px; color:#900;'>"
                f"<b>Configuration error:</b> {html.escape(str(error))}"
                "</div>"
            )

    def start_processing(_button=None) -> None:
        try:
            configuration = read_configuration()
            apply_configuration(configuration)
        except Exception:
            validate_configuration()
            return

        with output_widget:
            output_widget.clear_output()
            try:
                progress_path = run_all_country_batches_with_live_display(
                    country_codes=configuration["country_codes"],
                    year=configuration["year"],
                    batch_size=configuration["batch_size"],
                    delay_between_batches=configuration["batch_delay"],
                )
                if progress_path is None:
                    status_widget.value = (
                        "<div style='border:1px solid #d9a441; "
                        "padding:8px; color:#7a5200;'>"
                        "<b>Run finished without a validated country "
                        "output.</b><br>See the diagnostic row in the "
                        "output panel and processing log."
                        "</div>"
                    )
                else:
                    status_widget.value = (
                        "<div style='border:1px solid #7a9; padding:8px;'>"
                        "<b>Processing finished.</b><br>"
                        f"Global progress: {html.escape(str(progress_path))}"
                        "</div>"
                    )
            except Exception as error:
                status_widget.value = (
                    "<div style='border:1px solid #c66; "
                    "padding:8px; color:#900;'>"
                    f"<b>Processing error:</b> "
                    f"{html.escape(type(error).__name__ + ': ' + str(error))}"
                    "</div>"
                )
                traceback.print_exc()

    def generate_map_only(_button=None) -> None:
        try:
            configuration = read_configuration()
            apply_configuration(configuration)
        except Exception:
            validate_configuration()
            return

        with output_widget:
            output_widget.clear_output()
            try:
                create_global_map_outputs(configuration["year"])
            except FileNotFoundError as error:
                print(error)
                print(
                    "Run at least one country successfully before "
                    "generating a map."
                )
                print(
                    "Processing log:",
                    get_processing_log_path(configuration["year"]),
                )

    mode_widget.observe(update_mode_visibility, names="value")
    validate_button.on_click(validate_configuration)
    start_button.on_click(start_processing)
    map_only_button.on_click(generate_map_only)
    update_mode_visibility()

    controls = widgets.VBox(
        [
            widgets.HTML("<h3>Indicator 6.1.1 workflow controls</h3>"),
            mode_widget,
            year_widget,
            test_countries_widget,
            widgets.HBox([batch_size_widget, batch_delay_widget]),
            chunk_size_widget,
            minimum_coverage_widget,
            skip_completed_widget,
            checkpoints_widget,
            keep_archive_widget,
            keep_tiles_widget,
            keep_building_widget,
            generate_maps_widget,
            global_confirmation_widget,
            widgets.HBox([validate_button, start_button, map_only_button]),
            status_widget,
            output_widget,
        ]
    )
    return controls


## 30. Start or display the workflow controls

With `USE_WIDGET_DASHBOARD=True`, running the notebook only prepares functions and displays the dashboard; it does not automatically download or process data. With dashboard mode disabled, enabled tests may run in test mode, while the global runner starts only when `TEST_MODE=False`.


In [48]:
global_progress_path = None

if USE_WIDGET_DASHBOARD:
    workflow_dashboard = create_workflow_dashboard()
    display(workflow_dashboard)
    print(
        "Dashboard mode enabled: no analysis was started automatically."
    )

elif TEST_MODE:
    print("TEST_MODE=True: the global batch runner was not started.")
    print(
        "Enable one of the optional test switches near the top, or set "
        "TEST_MODE=False when ready for global processing."
    )

else:
    global_progress_path = run_all_country_batches_with_live_display(
        country_codes=RUN_COUNTRIES,
        year=ANALYSIS_YEAR,
        batch_size=5,
        delay_between_batches=5,
    )
    print("Global progress file:", global_progress_path)


Dashboard mode enabled: no analysis was started automatically.
